# House Prices — linear models, tree ensembles, and interpretation

The repository README explains how to download the Kaggle data and run this notebook.

Part I compares a median baseline, linear estimators and a single tree. Part II
adds five ensemble families, compares native and one-hot CatBoost categories,
and exhausts each small parameter grid with four search methods.

All model selection uses the training split and the same five folds. The primary
metric is **RMSLE**, matching the log-transformed target. MAE and RMSE are in USD;
R² is computed on original prices. Negative predicted prices are clipped to zero
consistently before computing metrics.


# 1. Setup

Import the tools and configure shared settings needed for data analysis, visualization, and model training.

In [ ]:
# Shared imports, CPU limits, and reproducibility settings.
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import hashlib
import importlib.metadata
import json
import platform
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import plotly.express as px
import seaborn as sns
from house_prices import features as house_features
from house_prices.data import load_house_sales
from house_prices.features import (
    DOMINANT_SHARE_THRESHOLD,
    add_house_features,
    build_engineered_pipeline,
    build_numeric_pipeline,
    build_preprocessor,
    select_categorical,
    select_numeric,
)
from catboost import CatBoostRegressor
from IPython.display import Markdown, display
from lightgbm import LGBMRegressor
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
    root_mean_squared_log_error,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    ParameterGrid,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.utils.validation import check_is_fitted
from skopt.acquisition import gaussian_ei
from threadpoolctl import threadpool_limits
from xgboost import XGBRegressor

SEED = 42
CV_JOBS = 8

threadpool_limits(limits=1)
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


def inverse_predictions(y_log):
    """Return nonnegative dollar predictions for every model and metric."""
    return np.expm1(np.maximum(np.asarray(y_log), 0.0))


package_versions = {
    name: importlib.metadata.version(name)
    for name in [
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "xgboost",
        "catboost",
        "lightgbm",
        "optuna",
        "scikit-optimize",
    ]
}
print(f"Python {platform.python_version()} | CPU | seed={SEED} | CV workers={CV_JOBS}")
display(pd.Series(package_versions, name="Version").to_frame())

Load the house sales data and recognize missing values so it is ready for analysis and modeling.

In [ ]:
# Jupyter normally starts this kernel in notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "house-prices.ipynb"
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"
df = load_house_sales(DATA_PATH)

Keep house identifiers for later error analysis while excluding them from the model inputs.

In [ ]:
# Drop the 'Id' column from the DataFrame and store it in a separate variable
ids = df["Id"].copy()
df = df.drop(["Id"], axis=1)

Reserve a holdout set to evaluate the final model on data that is not used during training or tuning.

In [ ]:
# Train test split
df, test = train_test_split(df, test_size=0.2, random_state=SEED)

# 2. EDA

Inspect column types and non-missing counts to understand the dataset structure and identify data quality issues.

In [ ]:
# Display information about the DataFrame
df.info()

Summarize numeric columns to understand typical values, ranges, and possible outliers.

In [ ]:
# Display summary statistics of the DataFrame
df.describe().round(3)

Measure missing-value rates to guide the choice of preprocessing strategies.

In [ ]:
# Display the percentage of missing values in each column
df.isna().mean().sort_values(ascending=False)

Check for duplicate rows that could distort the analysis and model evaluation.

In [ ]:
# Display the number of duplicated rows in the DataFrame
df.duplicated().sum()

Inspect the sale price distribution to assess its spread, skewness, and unusually expensive homes.

In [ ]:
# Show 'SalePrice' distribution plot
print(df["SalePrice"].describe())
plt.figure(figsize=(9, 8))
sns.histplot(df["SalePrice"], color="g", bins=100, alpha=0.4, kde=True)

Inspect log-transformed prices to see whether the transformation reduces target skewness before modeling.

In [ ]:
# Show log1p('SalePrice') distribution plot
print(np.log1p(df["SalePrice"]).describe())
plt.figure(figsize=(9, 8))
sns.histplot(np.log1p(df["SalePrice"]), color="g", bins=100, alpha=0.4, kde=True)

Use a boxplot to highlight the central price range and potential outliers.

In [ ]:
# Plot the training sale prices as a boxplot.
plt.figure(figsize=(10, 3))
sns.boxplot(x=df["SalePrice"], color="lightgreen")
plt.title("Sale price distribution")
plt.xlabel("SalePrice")
plt.show()

Use a boxplot of log-transformed prices to assess their spread and remaining potential outliers.

In [ ]:
# Plot the log1p-transformed training sale prices as a boxplot.
plt.figure(figsize=(10, 3))
sns.boxplot(x=np.log1p(df["SalePrice"]), color="lightgreen")
plt.title("Log-transformed Sale price distribution")
plt.xlabel("log(SalePrice + 1)")
plt.show()

Explore the price distribution interactively to inspect its shape and potential outliers in more detail.

In [ ]:
# Show an interactive price histogram with a marginal boxplot.
fig = px.histogram(
    df,
    x="SalePrice",
    marginal="box",
    nbins=100,
)
fig.show()

Explore log-transformed prices interactively to assess how the transformation changes the distribution and potential outliers.

In [ ]:
# Show an interactive log1p-transformed price histogram with a marginal boxplot.
fig = px.histogram(
    df,
    x=np.log1p(df["SalePrice"]),
    marginal="box",
    nbins=100,
)
fig.show()

Quantify price skewness and excess kurtosis to support the visual assessment of the target distribution.

In [ ]:
# Measure the skewness and excess kurtosis of training sale prices.
print(f"Skewness: {df['SalePrice'].skew():.3f}")
print(f"Excess kurtosis: {df['SalePrice'].kurt():.3f}")

- Skewness = 1.743 — **pronounced right skew**. Zero indicates no skewness by this measure. A positive value indicates a longer right tail: expensive homes form that tail and pull the mean above the median.
- Excess kurtosis = 5.475 — **pronounced extreme deviations**. A normal distribution has theoretical excess kurtosis of 0. Here it is substantially higher: prices far from the mean make a large contribution to this statistic.


Check which data types are present before separating numeric and categorical columns.

In [ ]:
# Numerical data distribution
list(set(df.dtypes.tolist()))

Collect numeric columns for a closer look at their distributions.

In [ ]:
# Numerical subset
df_num = df.select_dtypes(include=["float64", "int64"])

Inspect numeric distributions to spot skewed features, unusual ranges, and potential outliers.

In [ ]:
# Plot a histogram for each numeric training column.
df_num.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8);

- MSSubClass — nominal category code
- OverallQual — ordinal score; higher is better
- OverallCond — ordinal score; higher is better


Visualize Spearman correlations to identify monotonic relationships among numeric features and sale prices.

In [ ]:
# Select numeric columns and exclude the category code
numeric_df = df.select_dtypes(include="number")
numeric_df = numeric_df.drop(columns=["MSSubClass"])

# Keep SalePrice to examine relationships between features and price
corr = numeric_df.corr(method="spearman")

plt.figure(figsize=(24, 20))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.3,
    annot_kws={"size": 7},
    cbar_kws={"label": "Spearman correlation"},
)

plt.title("Feature and price correlations", fontsize=18, pad=20)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Identify strongly correlated feature pairs to spot potentially redundant information.

In [ ]:
# Correlations between features
feature_corr = corr.drop(index="SalePrice", columns="SalePrice")

# Keep each pair once and exclude the diagonal
mask = np.triu(np.ones(feature_corr.shape, dtype=bool), k=1)

feature_pairs = (
    feature_corr.where(mask)
    .stack()
    .dropna()
    .rename("Spearman")
    .rename_axis(["Feature 1", "Feature 2"])
    .reset_index()
)

feature_pairs["Abs"] = feature_pairs["Spearman"].abs()
feature_pairs = feature_pairs.sort_values("Abs", ascending=False).reset_index(drop=True)

display(feature_pairs[feature_pairs["Abs"] > 0.6].round(3))

Rank numeric features by their correlation with sale price to identify promising predictors.

In [ ]:
# Feature correlations with SalePrice
price_table = corr["SalePrice"].drop("SalePrice").dropna().rename("Spearman").to_frame()

price_table.index.name = "Feature"
price_table["Abs"] = price_table["Spearman"].abs()
price_table = price_table.sort_values("Abs", ascending=False)

display(price_table.round(3))

Inspect frequent feature values to identify columns dominated by a single value, including missing values.

In [ ]:
# Inspect the five most frequent exact values of each original training feature.
# Percentages include missing values and use all training rows as the denominator.
eda_features = df.drop(columns=["SalePrice"])
feature_columns = eda_features.columns.tolist()

for start in range(0, len(feature_columns), 12):
    batch = feature_columns[start : start + 12]
    nrows = (len(batch) + 3) // 4
    fig, axes = plt.subplots(nrows, 4, figsize=(18, 3.5 * nrows), squeeze=False)

    for ax, column in zip(axes.flat, batch):
        shares = (
            eda_features[column]
            .value_counts(dropna=False, normalize=True)
            .head(5)
            .mul(100)
        )
        labels = [
            "<missing>" if pd.isna(value) else str(value) for value in shares.index
        ]
        dominant_share = shares.iloc[0]
        ax.barh(labels, shares.to_numpy(), color="steelblue")
        ax.invert_yaxis()
        ax.axvline(
            100 * DOMINANT_SHARE_THRESHOLD, color="red", linestyle="--", linewidth=1
        )
        ax.set_xlim(0, 100)
        ax.set_xlabel("Share of all training rows, %")
        ax.set_title(
            f"{column}: most frequent = {dominant_share:.1f}%",
            fontsize=10,
            color="red"
            if dominant_share >= 100 * DOMINANT_SHARE_THRESHOLD
            else "black",
        )

    for ax in axes.flat[len(batch) :]:
        ax.remove()
    plt.tight_layout()
    plt.show()

# 3. Pipeline

Separate predictors from the target, retain original prices for evaluation, and log-transform the target to reduce skew during training.

In [ ]:
# Separate features, original prices, and the log1p target.
X = df.drop(columns=["SalePrice"])

# Keep original prices for metrics in dollars and the constant baseline.
y_price = df["SalePrice"].copy()
y = np.log1p(y_price)

Build reusable pipelines to engineer house features, handle missing values, scale numbers, encode categories, and filter columns dominated by one value.

In [ ]:
# Reuse the engineered features and fold-local preparation in the CLI.
numeric_pipeline = build_numeric_pipeline()
preprocessor = build_preprocessor()
base_pipeline = Pipeline([("preprocessor", clone(preprocessor))])
model = build_engineered_pipeline()
model

Build a feature-selection pipeline that keeps only numeric features with strong training Spearman correlations, including engineered features.

In [ ]:
class HighCorrelationSelector(TransformerMixin, BaseEstimator):
    """Keep numeric columns whose absolute training Spearman correlation exceeds a threshold."""

    def __init__(self, threshold=0.5):
        self.threshold = threshold

    def fit(self, X, y):
        numeric = X.select_dtypes(include="number").drop(
            columns=["MSSubClass"], errors="ignore"
        )
        # Ignore constant and entirely missing columns before computing correlations.
        numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
        # sklearn slices X and y by position; do not align y by its original labels.
        target = pd.Series(np.asarray(y), index=X.index)
        self.correlations_ = numeric.corrwith(target, method="spearman")
        self.selected_features_ = self.correlations_.index[
            self.correlations_.abs() > self.threshold
        ].tolist()
        if not self.selected_features_:
            raise ValueError(
                "No numeric features have an absolute Spearman correlation "
                f"greater than {self.threshold:g} with the training target."
            )
        self.n_features_in_ = X.shape[1]
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        check_is_fitted(self, "selected_features_")
        return X.loc[:, self.selected_features_].copy()

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "selected_features_")
        return np.asarray(self.selected_features_, dtype=object)


# Learn the supervised selection inside each training fold.
model_high_corr = Pipeline(
    [
        ("features", FunctionTransformer(add_house_features, validate=False)),
        ("selection", HighCorrelationSelector(threshold=0.5)),
        ("preprocessor", clone(preprocessor)),
    ]
)

model_high_corr

Preview the selected features on the training data; cross-validation will fit fresh selectors independently within each fold.

In [ ]:
# Fit a separate clone for inspection only, never for defining CV feature lists.
high_corr_preview = clone(model_high_corr).fit(X, y)
preview_selector = high_corr_preview.named_steps["selection"]
high_corr_table = (
    preview_selector.correlations_.loc[preview_selector.selected_features_]
    .rename("Spearman")
    .to_frame()
)
high_corr_table.index.name = "Feature"
high_corr_table["Abs"] = high_corr_table["Spearman"].abs()
high_corr_table = high_corr_table.sort_values("Abs", ascending=False)
print(f"Selected {len(high_corr_table)} numeric features on the full training split.")
display(high_corr_table.round(3))

# 4. Part I — Baseline and single estimators

## 4.1. Baseline

Establish shared cross-validation metrics and a median-price baseline to judge whether trained models add value.

In [ ]:
# Define CV metrics and evaluate the baseline that predicts the median price.


def metric_in_prices(y_true_log, y_pred_log, *, metric):
    """The model predicts log1p prices; the metric receives prices in dollars."""
    return metric(np.expm1(y_true_log), inverse_predictions(y_pred_log))


# Scorers for models trained on y = log1p(SalePrice).
scoring = {
    "MAE": make_scorer(
        metric_in_prices, metric=mean_absolute_error, greater_is_better=False
    ),
    "RMSE": make_scorer(
        metric_in_prices, metric=root_mean_squared_error, greater_is_better=False
    ),
    "R2": make_scorer(metric_in_prices, metric=r2_score),
    "RMSLE": make_scorer(
        metric_in_prices, metric=root_mean_squared_log_error, greater_is_better=False
    ),
}

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_splits = list(cv.split(X, y))

# The baseline predicts the median original price.
baseline = DummyRegressor(strategy="median")
baseline_scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "RMSLE": "neg_root_mean_squared_log_error",
}

baseline_scores = cross_validate(
    baseline,
    X,
    y_price,
    cv=cv_splits,
    scoring=baseline_scoring,
    n_jobs=CV_JOBS,
    error_score="raise",
)

baseline_folds = pd.DataFrame(
    {
        "MAE": -baseline_scores["test_MAE"],
        "RMSE": -baseline_scores["test_RMSE"],
        "R2": baseline_scores["test_R2"],
        "RMSLE": -baseline_scores["test_RMSLE"],
    }
)

baseline_summary = baseline_folds.agg(["mean", "std"]).T
baseline_summary.columns = ["CV mean", "CV std"]

baseline_summary.round(3)

## 4.2. LinearRegression

Compare linear regression across the base, engineered, and high-correlation feature sets using identical cross-validation folds and metrics.

In [ ]:
# Compare the three preprocessing variants using the same cross-validation folds.
def evaluate_cv(estimator):
    scores = cross_validate(
        estimator,
        X,
        y,
        cv=cv_splits,
        scoring=scoring,
        n_jobs=CV_JOBS,
        error_score="raise",
    )
    folds = pd.DataFrame(
        {
            "MAE": -scores["test_MAE"],
            "RMSE": -scores["test_RMSE"],
            "R2": scores["test_R2"],
            "RMSLE": -scores["test_RMSLE"],
        }
    )
    summary = folds.agg(["mean", "std"]).T
    summary.columns = ["CV mean", "CV std"]
    return folds, summary, np.asarray(scores["fit_time"])


preparations = {"base": base_pipeline, "FE": model, "high_corr": model_high_corr}
preparation_labels = {
    "base": "base_pipeline",
    "FE": "model",
    "high_corr": "model_high_corr",
}
model_candidates = {}
cv_folds = {"Median baseline": baseline_folds}
cv_summaries = {"Median baseline": baseline_summary}
cv_fit_times = {"Median baseline": np.asarray(baseline_scores["fit_time"])}
search_times = {"Median baseline": 0.0}
model_metadata = {
    "Median baseline": {
        "part": "I",
        "family": "Median baseline",
        "preparation": "none",
        "encoding": "none",
        "parameters": {"strategy": "median"},
        "method": "none",
    }
}


def make_model_variants(regressor):
    return {
        variant: Pipeline(
            [("preparation", clone(preparation)), ("regressor", clone(regressor))]
        )
        for variant, preparation in preparations.items()
    }


def evaluate_family(models, family):
    for variant, estimator in models.items():
        name = f"{family} / {variant}"
        model_candidates[name] = estimator
        cv_folds[name], cv_summaries[name], cv_fit_times[name] = evaluate_cv(estimator)
        search_times.setdefault(name, 0.0)
        model_metadata[name] = {
            "part": "I",
            "family": family,
            "preparation": variant,
            "encoding": "numeric" if variant == "high_corr" else "OHE",
            "parameters": estimator.named_steps["regressor"].get_params(deep=False),
            "method": "GridSearchCV" if family != "LinearRegression" else "none",
        }
    return pd.concat(cv_summaries, names=["Model", "Metric"])


linear_models = make_model_variants(LinearRegression())
comparison = evaluate_family(linear_models, "LinearRegression")
comparison.round(3)

## 4.3. Ridge

Evaluate Ridge regularization across all three feature sets before tuning its strength.

In [ ]:
# Compare Ridge on all three preprocessing variants before tuning.
ridge_models = make_model_variants(Ridge(alpha=1.0, random_state=SEED))
comparison = evaluate_family(ridge_models, "Ridge")
comparison.round(3)

Choose Ridge regularization separately for each feature set using the same alpha grid, and compare training and validation errors.

In [ ]:
# Tune Ridge alpha separately for each preprocessing variant.
alphas = [0.01, 0.1, 1, 3, 10, 30, 100, 1000]
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for (name, ridge), ax in zip(ridge_models.items(), axes):
    search_start = perf_counter()
    search = GridSearchCV(
        clone(ridge),
        {"regressor__alpha": alphas},
        cv=cv_splits,
        scoring=scoring["RMSLE"],
        refit=False,
        return_train_score=True,
        n_jobs=CV_JOBS,
        error_score="raise",
    )
    search.fit(X, y)
    train_errors = -search.cv_results_["mean_train_score"]
    cv_errors = -search.cv_results_["mean_test_score"]

    search_times[f"Ridge / {name}"] = perf_counter() - search_start
    best_alpha = alphas[np.argmin(cv_errors)]
    ridge.set_params(regressor__alpha=best_alpha)
    print(
        f"Ridge / {name}: best alpha in the tested grid = {best_alpha:g}, CV RMSLE = {min(cv_errors):.4f}"
    )
    ax.plot(alphas, train_errors, "o-", label="Train")
    ax.plot(alphas, cv_errors, "o-", label="CV")
    ax.axvline(best_alpha, color="green", linestyle="--", label=f"alpha={best_alpha:g}")
    ax.set(xscale="log", xlabel="alpha", ylabel="RMSLE", title=f"Ridge / {name}")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()
comparison = evaluate_family(ridge_models, "Ridge")

## 4.4. Lasso

Evaluate Lasso regularization across all three feature sets before tuning its strength.

In [ ]:
# Compare Lasso on all three preprocessing variants before tuning.
lasso_models = make_model_variants(
    Lasso(alpha=100.0, max_iter=20000, random_state=SEED)
)
comparison = evaluate_family(lasso_models, "Lasso")
comparison.round(3)

Choose Lasso regularization separately for each feature set using the same alpha grid, and compare training and validation errors.

In [ ]:
# Tune Lasso alpha separately for each preprocessing variant.
alphas = [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 1, 10, 100]
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for (name, lasso), ax in zip(lasso_models.items(), axes):
    search_start = perf_counter()
    search = GridSearchCV(
        clone(lasso),
        {"regressor__alpha": alphas},
        cv=cv_splits,
        scoring=scoring["RMSLE"],
        refit=False,
        return_train_score=True,
        n_jobs=CV_JOBS,
        error_score="raise",
    )
    search.fit(X, y)
    train_errors = -search.cv_results_["mean_train_score"]
    cv_errors = -search.cv_results_["mean_test_score"]

    search_times[f"Lasso / {name}"] = perf_counter() - search_start
    best_alpha = alphas[np.argmin(cv_errors)]
    lasso.set_params(regressor__alpha=best_alpha)
    print(
        f"Lasso / {name}: best alpha in the tested grid = {best_alpha:g}, CV RMSLE = {min(cv_errors):.4f}"
    )
    ax.plot(alphas, train_errors, "o-", label="Train")
    ax.plot(alphas, cv_errors, "o-", label="CV")
    ax.axvline(best_alpha, color="green", linestyle="--", label=f"alpha={best_alpha:g}")
    ax.set(xscale="log", xlabel="alpha", ylabel="RMSLE", title=f"Lasso / {name}")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()
comparison = evaluate_family(lasso_models, "Lasso")

## 4.5. DecisionTreeRegressor

Compare nonlinear decision trees across all three feature sets before tuning their complexity.

In [ ]:
# Compare decision trees on all three preprocessing variants before tuning.
tree_models = make_model_variants(
    DecisionTreeRegressor(max_depth=32, min_samples_leaf=4, random_state=SEED)
)
comparison = evaluate_family(tree_models, "DecisionTree")
comparison.round(3)

Tune tree depth and leaf size for all three feature sets, then compare the baseline and all twelve models using cross-validation.

In [ ]:
# Search the same tree depth and leaf size grid for each variant.
param_grid = {
    "regressor__max_depth": [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 16, 24, 32, 64, 128],
    "regressor__min_samples_leaf": [1, 2, 4, 8, 10, 16, 20, 32],
}
tree_grids = {}
for name, tree in tree_models.items():
    search_start = perf_counter()
    search = GridSearchCV(
        clone(tree),
        param_grid,
        cv=cv_splits,
        scoring=scoring["RMSLE"],
        refit=False,
        n_jobs=CV_JOBS,
        return_train_score=True,
        error_score="raise",
    )
    search.fit(X, y)
    search_times[f"DecisionTree / {name}"] = perf_counter() - search_start
    tree.set_params(**search.best_params_)
    print(
        f"DecisionTree / {name}: {search.best_params_}, CV RMSLE = {-search.best_score_:.4f}"
    )
    results = pd.DataFrame(search.cv_results_)
    results["max_depth"] = results["param_regressor__max_depth"].astype(int)
    results["min_samples_leaf"] = results["param_regressor__min_samples_leaf"].astype(
        int
    )
    results["CV RMSLE"] = -results["mean_test_score"]
    tree_grids[name] = results.pivot(
        index="max_depth", columns="min_samples_leaf", values="CV RMSLE"
    )

# A common color scale makes all three validation-error maps comparable.
vmin = min(grid.to_numpy().min() for grid in tree_grids.values())
vmax = max(grid.to_numpy().max() for grid in tree_grids.values())
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for (name, grid), ax in zip(tree_grids.items(), axes):
    sns.heatmap(
        grid,
        annot=True,
        fmt=".3f",
        cmap="YlGnBu_r",
        vmin=vmin,
        vmax=vmax,
        ax=ax,
        cbar_kws={"label": "CV RMSLE — lower is better"},
    )
    ax.set_title(f"DecisionTree / {name}")
    ax.tick_params(axis="both", labelrotation=0)
plt.tight_layout()
plt.show()

comparison = evaluate_family(tree_models, "DecisionTree")
display(comparison.round(3))
display(comparison.xs("RMSLE", level="Metric").sort_values("CV mean").round(4))

# 5. Part II — Tree ensembles and gradient boosting

We compare Random Forest, scikit-learn Gradient Boosting, XGBoost, CatBoost,
and LightGBM on the same training rows and five CV folds as Part I. Models
predict `log1p(SalePrice)`; **mean CV RMSLE is the optimization objective**.
MAE and RMSE are reported in USD, and R² is calculated on original prices.
All preprocessing, including the 86% dominant-value filter and supervised
feature selection, is learned inside each training fold. The holdout is not
used anywhere in Part II.

Each model has a deliberately small, finite space of **32 configurations**.
Every space is exhausted independently by GridSearchCV, RandomizedSearchCV,
Optuna TPE followed by completion of missing configurations, and Bayesian
optimization with a Gaussian Process and Expected Improvement. Exhausting
identical spaces should produce the same optimum; the useful comparison is
how quickly each search discovers it. This exercise demonstrates search
behaviour rather than claiming that one optimizer has better final accuracy.

We retain `base`, `FE`, and `high_corr` preprocessing branches. CatBoost uses
native categories and one-hot encoding on both branches that contain
categories. The `high_corr` branch contains numeric features only, so there
is one CatBoost run for that branch. Thus, there are **17 model variants,
68 independent exhaustive searches, and 10,880 CV fits** in Part II, before
small winner-validation and interpretation runs. All estimators use one CPU
thread; CV uses up to eight workers; individual estimators use one thread.

## 5.1. Native CatBoost and controlled encoding comparison

The native and one-hot CatBoost branches share feature engineering, the
fold-local dominant-value filter, numeric imputation, missing indicators,
and scaling. They also share exactly the same hyperparameter configurations,
folds, and seeds. Only categorical representation changes. Native CatBoost
receives missing categories as the string `Missing`, and receives `MSSubClass`
as a nominal category. Its category statistics are learned inside each fit,
without access to CV validation targets. This compares the complete encoding
strategies under a matched budget; native CatBoost may itself one-hot encode
some low-cardinality features internally.

In [ ]:
import importlib.metadata as _package_metadata
import warnings as _warnings
from sklearn.exceptions import ConvergenceWarning


class NativeCatBoostPreprocessor(TransformerMixin, BaseEstimator):
    """Match the OHE preparation while keeping categorical columns as strings."""

    def fit(self, X, y=None):
        self.numeric_columns_ = list(select_numeric(X))
        self.categorical_columns_ = list(select_categorical(X))
        self.numeric_pipeline_ = clone(numeric_pipeline)
        self.numeric_pipeline_.fit(X.loc[:, self.numeric_columns_], y)
        numeric_names = list(
            self.numeric_pipeline_.get_feature_names_out(self.numeric_columns_)
        )
        self.output_columns_ = numeric_names + self.categorical_columns_
        imputer = self.numeric_pipeline_.named_steps["imputer"]
        indicator_parents = (
            [self.numeric_columns_[i] for i in imputer.indicator_.features_]
            if imputer.add_indicator
            else []
        )
        self.source_columns_ = (
            self.numeric_columns_ + indicator_parents + self.categorical_columns_
        )
        if len(self.output_columns_) != len(self.source_columns_):
            raise AssertionError("Native CatBoost feature mapping is inconsistent.")
        self.n_features_in_ = X.shape[1]
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        check_is_fitted(self, "output_columns_")
        numeric_values = self.numeric_pipeline_.transform(
            X.loc[:, self.numeric_columns_]
        )
        result = pd.DataFrame(
            numeric_values,
            columns=self.output_columns_[: numeric_values.shape[1]],
            index=X.index,
        )
        for column in self.categorical_columns_:
            # Convert before concatenating so numeric nominal codes are categorical.
            result[column] = X[column].astype("string").fillna("Missing").astype(object)
        return result.loc[:, self.output_columns_]

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "output_columns_")
        return np.asarray(self.output_columns_, dtype=object)


class NativeCatBoostRegressor(RegressorMixin, BaseEstimator):
    """Infer CatBoost categorical names after fold-local preprocessing."""

    def __init__(
        self,
        iterations=300,
        depth=4,
        learning_rate=0.03,
        l2_leaf_reg=3,
        subsample=0.8,
        random_seed=42,
        thread_count=1,
    ):
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.l2_leaf_reg = l2_leaf_reg
        self.subsample = subsample
        self.random_seed = random_seed
        self.thread_count = thread_count

    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Native CatBoost requires the prepared DataFrame.")
        self.cat_features_ = list(X.select_dtypes(exclude="number").columns)
        self.model_ = CatBoostRegressor(
            iterations=self.iterations,
            depth=self.depth,
            learning_rate=self.learning_rate,
            l2_leaf_reg=self.l2_leaf_reg,
            subsample=self.subsample,
            random_seed=self.random_seed,
            thread_count=self.thread_count,
            bootstrap_type="Bernoulli",
            loss_function="RMSE",
            task_type="CPU",
            allow_writing_files=False,
            verbose=False,
        )
        self.model_.fit(X, y, cat_features=self.cat_features_)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]
        return self

    def predict(self, X):
        check_is_fitted(self, "model_")
        return self.model_.predict(X)

    @property
    def feature_importances_(self):
        check_is_fitted(self, "model_")
        return self.model_.feature_importances_


def make_native_preparation(variant):
    steps = []
    if variant == "FE":
        steps.append(
            ("features", FunctionTransformer(add_house_features, validate=False))
        )
    elif variant != "base":
        raise ValueError("Native categorical comparison uses base or FE.")
    steps.append(("preprocessor", NativeCatBoostPreprocessor()))
    return Pipeline(steps)

## 5.2. Small, justified hyperparameter spaces

For about 1,168 training houses, shallow boosting trees and minimum leaf sizes
limit overfitting. Two tree counts and learning rates cover a practical
underfitting/overfitting trade-off without turning the notebook into a large
benchmark. Row subsampling adds stochastic regularization. Random Forest also
varies feature subsampling; CatBoost varies L2 regularization; LightGBM limits
leaf counts. XGBoost fixes column subsampling at 0.8 and L2 regularization at
5; LightGBM fixes column subsampling at 0.8 and L2 regularization at 1.

Random Forest uses bootstrap sampling with `max_samples`; this samples rows
with replacement. The boosting models' `subsample` controls the fraction of
rows used for tree building. CatBoost explicitly uses Bernoulli bootstrap,
and LightGBM sets `subsample_freq=1`, so these subsampling parameters are
active. No early stopping uses the holdout or CV validation labels.

In [ ]:
ensemble_spaces = {
    "RandomForest": {
        "regressor__n_estimators": [200, 400],
        "regressor__max_depth": [8, None],
        "regressor__min_samples_leaf": [1, 4],
        "regressor__max_features": [0.5, 1.0],
        "regressor__max_samples": [0.7, 1.0],
    },
    "GradientBoosting": {
        "regressor__n_estimators": [200, 500],
        "regressor__max_depth": [2, 3],
        "regressor__min_samples_leaf": [3, 8],
        "regressor__learning_rate": [0.03, 0.1],
        "regressor__subsample": [0.8, 1.0],
    },
    "XGBoost": {
        "regressor__n_estimators": [300, 600],
        "regressor__max_depth": [2, 4],
        "regressor__min_child_weight": [1, 5],
        "regressor__learning_rate": [0.03, 0.1],
        "regressor__subsample": [0.8, 1.0],
    },
    "CatBoost": {
        "regressor__iterations": [300, 600],
        "regressor__depth": [4, 6],
        "regressor__l2_leaf_reg": [3, 10],
        "regressor__learning_rate": [0.03, 0.1],
        "regressor__subsample": [0.8, 1.0],
    },
    "LightGBM": {
        "regressor__n_estimators": [300, 600],
        "regressor__num_leaves": [7, 15],
        "regressor__min_child_samples": [10, 30],
        "regressor__learning_rate": [0.03, 0.1],
        "regressor__subsample": [0.8, 1.0],
    },
}

ensemble_regressors = {
    "RandomForest": RandomForestRegressor(bootstrap=True, random_state=SEED, n_jobs=1),
    "GradientBoosting": GradientBoostingRegressor(random_state=SEED),
    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        colsample_bytree=0.8,
        reg_lambda=5,
        importance_type="total_gain",
        random_state=SEED,
        n_jobs=1,
        verbosity=0,
    ),
    "CatBoost": CatBoostRegressor(
        loss_function="RMSE",
        bootstrap_type="Bernoulli",
        task_type="CPU",
        random_seed=SEED,
        thread_count=1,
        allow_writing_files=False,
        verbose=False,
    ),
    "LightGBM": LGBMRegressor(
        objective="regression",
        colsample_bytree=0.8,
        reg_lambda=1,
        subsample_freq=1,
        importance_type="gain",
        deterministic=True,
        force_col_wise=True,
        random_state=SEED,
        n_jobs=1,
        verbosity=-1,
    ),
}

ensemble_variants = {}
for family, regressor in ensemble_regressors.items():
    for variant, preparation in preparations.items():
        encoding = "numeric" if variant == "high_corr" else "one-hot"
        suffix = " OHE" if family == "CatBoost" and encoding == "one-hot" else ""
        name = f"{family}{suffix} / {variant}"
        ensemble_variants[name] = {
            "estimator": Pipeline(
                [
                    ("preparation", clone(preparation)),
                    ("regressor", clone(regressor)),
                ]
            ),
            "family": family,
            "preparation": variant,
            "encoding": encoding,
            "grid": ensemble_spaces[family],
        }

for variant in ["base", "FE"]:
    name = f"CatBoost native / {variant}"
    ensemble_variants[name] = {
        "estimator": Pipeline(
            [
                ("preparation", make_native_preparation(variant)),
                ("regressor", NativeCatBoostRegressor(random_seed=SEED)),
            ]
        ),
        "family": "CatBoost",
        "preparation": variant,
        "encoding": "native",
        "grid": ensemble_spaces["CatBoost"],
    }

assert len(ensemble_variants) == 17
assert all(len(list(ParameterGrid(grid))) == 32 for grid in ensemble_spaces.values())
display(
    pd.DataFrame(
        [
            {
                "Family": family,
                "Configurations": len(list(ParameterGrid(grid))),
                "Search space": {
                    key.removeprefix("regressor__"): values
                    for key, values in grid.items()
                },
            }
            for family, grid in ensemble_spaces.items()
        ]
    ).set_index("Family")
)

## 5.3. Exhaustive search engine and resumable results

Every method evaluates every configuration with exactly the same train/CV
protocol. `RandomizedSearchCV(n_iter=32)` samples list-valued spaces without
replacement. Optuna first makes 32 TPE proposals (`n_startup_trials=5`), reuses
only its **own** duplicate evaluations, then enqueues every unseen
configuration. Thus duplicate proposals never trigger duplicate fitting.
The Bayesian method starts with five unique seeded configurations and then
maximizes Gaussian-process Expected Improvement over the remaining finite
pool. Removing each evaluated point guarantees termination and full coverage.

Completed independent searches are cached locally under `.cache/`. Keys
include the training data, CV splits, marked experiment-source fingerprint,
estimator/preprocessing parameters, parameter space, seed, and exact package
versions. A restarted run reuses only completed, compatible searches; an
interrupted search is rerun. Each method fits independently of other methods.
The cache is an execution aid, not an additional dataset or a saved holdout.

Convergence time plots use **cumulative summed CV worker fit + score time**.
This measures evaluation effort and ignores overlapping folds. It is not
elapsed wall time. The audit separately reports measured search wall time,
including optimizer overhead. Cache hits retain original measurements.

In [ ]:
SEARCH_METHODS = (
    "GridSearchCV",
    "RandomizedSearchCV",
    "Optuna TPE + completion",
    "Bayesian GP + EI",
)
_SEARCH_ENGINE_VERSION = "house-prices-exhaustive-v3"
_SEARCH_METRICS = ("MAE", "RMSE", "R2", "RMSLE")
_SEARCH_CACHE = PROJECT_ROOT / ".cache" / "ensemble-searches"
_SEARCH_CACHE.mkdir(parents=True, exist_ok=True)


def _config_key(params):
    return tuple((name, params[name]) for name in sorted(params))


def _source_fingerprint():
    # Hash marked experiment cells; adding report plots does not invalidate fits.
    document = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    sources = [
        "".join(cell["source"])
        for cell in document["cells"]
        if cell["cell_type"] == "code"
        and "experiment-source" in cell.get("metadata", {}).get("tags", [])
    ]
    if not sources:
        raise RuntimeError(
            "Missing experiment-source tags: cannot validate cached fits."
        )
    sources.append(Path(house_features.__file__).read_text(encoding="utf-8"))
    return hashlib.sha256("\n\n".join(sources).encode()).hexdigest()


def _search_fingerprint(name, specification, method):
    versions = {
        package: _package_metadata.version(package)
        for package in [
            "numpy",
            "pandas",
            "scipy",
            "scikit-learn",
            "xgboost",
            "catboost",
            "lightgbm",
            "optuna",
            "scikit-optimize",
            "joblib",
        ]
    }
    return joblib.hash(
        {
            "engine": _SEARCH_ENGINE_VERSION,
            "source": _source_fingerprint(),
            "data": (X, y, y_price),
            "folds": cv_splits,
            "name": name,
            "method": method,
            "seed": SEED,
            "cv_jobs": CV_JOBS,
            "estimator": specification["estimator"],
            "grid": specification["grid"],
            "versions": versions,
        }
    )


def _record_from_scores(params, config_id, scores, sequence, trial_number=None):
    folds = pd.DataFrame(
        {
            metric: (
                scores[f"test_{metric}"]
                if metric == "R2"
                else -np.asarray(scores[f"test_{metric}"])
            )
            for metric in _SEARCH_METRICS
        }
    )
    return {
        "params": dict(params),
        "config_id": int(config_id),
        "sequence": int(sequence),
        "trial_number": trial_number,
        "folds": folds,
        "mean_rmsle": float(folds["RMSLE"].mean()),
        "mean_fit_seconds": float(np.mean(scores["fit_time"])),
        "compute_seconds": float(
            np.sum(scores["fit_time"]) + np.sum(scores["score_time"])
        ),
    }


def _evaluate_search_configuration(
    estimator, params, config_id, sequence, trial_number=None
):
    scores = cross_validate(
        clone(estimator).set_params(**params),
        X,
        y,
        scoring=scoring,
        cv=cv_splits,
        n_jobs=CV_JOBS,
        error_score="raise",
    )
    return _record_from_scores(params, config_id, scores, sequence, trial_number)


def _records_from_searchcv(search, candidates):
    ids = {_config_key(params): i for i, params in enumerate(candidates)}
    results = search.cv_results_
    records = []
    for sequence, params in enumerate(results["params"]):
        # SearchCV exposes mean fit/score times, whose fold sums equal n_splits * mean.
        scores = {
            f"test_{metric}": np.asarray(
                [
                    results[f"split{fold}_test_{metric}"][sequence]
                    for fold in range(len(cv_splits))
                ]
            )
            for metric in _SEARCH_METRICS
        }
        scores["fit_time"] = np.full(len(cv_splits), results["mean_fit_time"][sequence])
        scores["score_time"] = np.full(
            len(cv_splits), results["mean_score_time"][sequence]
        )
        records.append(
            _record_from_scores(params, ids[_config_key(params)], scores, sequence)
        )
    return records


def _validate_search_payload(payload, candidates):
    records = payload["records"]
    expected = {_config_key(params) for params in candidates}
    observed = [_config_key(record["params"]) for record in records]
    assert len(observed) == len(expected) == len(set(observed))
    assert set(observed) == expected
    assert sorted(record["config_id"] for record in records) == list(
        range(len(candidates))
    )
    for record in records:
        assert record["folds"].shape == (len(cv_splits), len(_SEARCH_METRICS))
        assert np.isfinite(record["folds"].to_numpy()).all()
        assert np.isfinite(record["compute_seconds"]) and record["compute_seconds"] >= 0
    return payload


def run_exhaustive_search(name, specification, method):
    candidates = list(ParameterGrid(specification["grid"]))
    config_ids = {_config_key(params): i for i, params in enumerate(candidates)}
    fingerprint = _search_fingerprint(name, specification, method)
    cache_path = _SEARCH_CACHE / f"{fingerprint}.joblib"
    if cache_path.exists():
        try:
            payload = joblib.load(cache_path)
            assert payload["fingerprint"] == fingerprint
            _validate_search_payload(payload, candidates)
            payload["cache_hit"] = True
            return payload
        except (OSError, EOFError, ValueError, KeyError, AssertionError) as error:
            print(f"Ignoring incomplete/incompatible local cache: {error}")

    estimator, grid = specification["estimator"], specification["grid"]
    start = perf_counter()
    duplicate_proposals = 0
    total_trials = len(candidates)
    if method in ("GridSearchCV", "RandomizedSearchCV"):
        common = dict(
            estimator=clone(estimator),
            cv=cv_splits,
            scoring=scoring,
            refit=False,
            n_jobs=CV_JOBS,
            error_score="raise",
            return_train_score=False,
            pre_dispatch=CV_JOBS,
        )
        if method == "GridSearchCV":
            search = GridSearchCV(param_grid=grid, **common)
        else:
            search = RandomizedSearchCV(
                param_distributions=grid,
                n_iter=len(candidates),
                random_state=SEED,
                **common,
            )
        search.fit(X, y)
        records = _records_from_searchcv(search, candidates)

    elif method == "Optuna TPE + completion":
        unique_records = {}
        records = []
        sampler = optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5)
        study = optuna.create_study(direction="minimize", sampler=sampler)

        def objective(trial):
            params = {
                key: trial.suggest_categorical(key, values)
                for key, values in sorted(grid.items())
            }
            key = _config_key(params)
            if key not in unique_records:
                record = _evaluate_search_configuration(
                    estimator, params, config_ids[key], len(records), trial.number
                )
                unique_records[key] = record
                records.append(record)
            return unique_records[key]["mean_rmsle"]

        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study.optimize(objective, n_trials=len(candidates), n_jobs=1)
        missing = [
            params for params in candidates if _config_key(params) not in unique_records
        ]
        for params in missing:
            study.enqueue_trial(params)
        if missing:
            study.optimize(objective, n_trials=len(missing), n_jobs=1)
        total_trials = len(study.trials)
        duplicate_proposals = total_trials - len(records)

    elif method == "Bayesian GP + EI":
        keys = sorted(grid)
        coordinates = np.asarray(
            [[grid[key].index(params[key]) for key in keys] for params in candidates],
            dtype=float,
        )
        rng = np.random.default_rng(SEED)
        initial = list(rng.choice(len(candidates), size=5, replace=False))
        remaining = set(range(len(candidates)))
        records = []
        for config_id in initial:
            records.append(
                _evaluate_search_configuration(
                    estimator, candidates[config_id], config_id, len(records)
                )
            )
            remaining.remove(config_id)
        while remaining:
            observed_ids = [record["config_id"] for record in records]
            losses = np.asarray([record["mean_rmsle"] for record in records])
            gp = GaussianProcessRegressor(
                kernel=ConstantKernel(1.0, (0.1, 10.0))
                * Matern(
                    length_scale=np.ones(len(keys)),
                    length_scale_bounds=(0.01, 100.0),
                    nu=2.5,
                ),
                alpha=1e-8,
                normalize_y=True,
                random_state=SEED,
                n_restarts_optimizer=0,
            )
            with _warnings.catch_warnings():
                _warnings.simplefilter("ignore", ConvergenceWarning)
                gp.fit(coordinates[observed_ids], losses)
            remaining_ids = sorted(remaining)
            acquisition = gaussian_ei(
                coordinates[remaining_ids], gp, y_opt=float(losses.min()), xi=0.001
            )
            assert np.isfinite(acquisition).all()
            config_id = remaining_ids[int(np.argmax(acquisition))]
            records.append(
                _evaluate_search_configuration(
                    estimator, candidates[config_id], config_id, len(records)
                )
            )
            remaining.remove(config_id)
    else:
        raise ValueError(f"Unknown search method: {method}")

    payload = {
        "fingerprint": fingerprint,
        "model": name,
        "method": method,
        "records": records,
        "search_seconds": perf_counter() - start,
        "duplicate_proposals": duplicate_proposals,
        "total_trials": total_trials,
        "cache_hit": False,
    }
    _validate_search_payload(payload, candidates)
    # Round only the tie-break key; retain full precision in reported scores.
    best = min(
        records, key=lambda row: (round(row["mean_rmsle"], 12), row["config_id"])
    )
    payload["best_config_id"] = best["config_id"]
    temporary_path = cache_path.with_suffix(".tmp")
    joblib.dump(payload, temporary_path)
    temporary_path.replace(cache_path)
    return payload

## 5.4. Run and audit all searches

The progress messages identify each independently completed search. The
coverage assertions require exactly 32 unique evaluated configurations per
method and variant. A second assertion compares all fold metrics for the
same configurations across methods. Search time is measured separately from
a fresh CV evaluation of each variant's winner, which provides directly
comparable fit times for the final model table. The sample standard deviation
of five fold metrics describes fold variability, not a confidence interval.

In [ ]:
ensemble_search_results = {}
ensemble_audit_rows = []
ensemble_winner_rows = []

for name, specification in ensemble_variants.items():
    method_results = {}
    for method in SEARCH_METHODS:
        print(f"{name}: {method}", flush=True)
        result = run_exhaustive_search(name, specification, method)
        ensemble_search_results[(name, method)] = result
        method_results[method] = result
        best_record = next(
            record
            for record in result["records"]
            if record["config_id"] == result["best_config_id"]
        )
        ensemble_audit_rows.append(
            {
                "Model": name,
                "Family": specification["family"],
                "Preparation": specification["preparation"],
                "Encoding": specification["encoding"],
                "Method": method,
                "Expected configs": 32,
                "Unique configs": len(result["records"]),
                "Coverage": len(result["records"]) / 32,
                "Total proposals": result["total_trials"],
                "Duplicate proposals": result["duplicate_proposals"],
                "Best config ID": result["best_config_id"],
                "Best CV RMSLE": best_record["mean_rmsle"],
                "Search wall seconds": result["search_seconds"],
                "Summed CV worker seconds": sum(
                    row["compute_seconds"] for row in result["records"]
                ),
                "Cache hit": result["cache_hit"],
                "Parameters": best_record["params"],
            }
        )
        print(
            f"  32/32 unique; CV RMSLE={best_record['mean_rmsle']:.5f}; "
            f"{result['search_seconds']:.1f}s; cache={result['cache_hit']}",
            flush=True,
        )

    reference = method_results["GridSearchCV"]
    reference_by_id = {row["config_id"]: row for row in reference["records"]}
    for method, result in method_results.items():
        assert result["best_config_id"] == reference["best_config_id"], (name, method)
        for row in result["records"]:
            np.testing.assert_allclose(
                row["folds"].to_numpy(),
                reference_by_id[row["config_id"]]["folds"].to_numpy(),
                rtol=1e-9,
                atol=1e-10,
                err_msg=f"Independent methods disagree: {name}, {method}",
            )

    winner = reference_by_id[reference["best_config_id"]]
    tuned = clone(specification["estimator"]).set_params(**winner["params"])
    # An independent same-fold evaluation supplies actual per-fold fit times.
    folds, summary, fit_times = evaluate_cv(tuned)
    np.testing.assert_allclose(
        folds.to_numpy(), winner["folds"].to_numpy(), rtol=1e-9, atol=1e-10
    )
    model_candidates[name] = tuned
    cv_folds[name], cv_summaries[name] = folds, summary
    cv_fit_times[name] = np.asarray(fit_times)
    search_times[name] = sum(row["search_seconds"] for row in method_results.values())
    model_metadata[name] = {
        "part": "II",
        "family": specification["family"],
        "preparation": specification["preparation"],
        "encoding": specification["encoding"],
        "parameters": {
            key.removeprefix("regressor__"): value
            for key, value in winner["params"].items()
        },
        "method": "Four exhaustive searches; identical optimum",
    }
    ensemble_winner_rows.append(
        {
            "Model": name,
            "CV RMSLE": summary.loc["RMSLE", "CV mean"],
            "CV RMSLE std": summary.loc["RMSLE", "CV std"],
            "Mean CV fit seconds": np.mean(fit_times),
            "Parameters": model_metadata[name]["parameters"],
        }
    )

ensemble_audit = pd.DataFrame(ensemble_audit_rows)
ensemble_winners = pd.DataFrame(ensemble_winner_rows).set_index("Model")
assert ensemble_audit.shape[0] == 68
assert ensemble_audit["Coverage"].eq(1).all()
assert len(ensemble_winners) == 17
print("All 68 searches exhausted their 32-configuration spaces.")
display(
    ensemble_audit[
        [
            "Model",
            "Method",
            "Unique configs",
            "Coverage",
            "Duplicate proposals",
            "Best CV RMSLE",
            "Search wall seconds",
            "Cache hit",
        ]
    ].round(5)
)
display(ensemble_winners.sort_values("CV RMSLE").round(5))

## 5.5. Search convergence

Each row below corresponds to one fixed model and preprocessing variant.
The left panel tracks the best observed CV RMSLE against the number of unique
configurations evaluated. The right panel uses cumulative CV worker time,
which excludes Gaussian-process/TPE overhead and does not account for parallel
fold overlap. Use the audit's measured wall time for total runtime comparisons.
At complete coverage every method reaches the same optimum by construction.

For GridSearchCV and RandomizedSearchCV, the configuration axis follows the stored parameter-list order, not observed completion timestamps of parallel fits. The panels below separately compare total measured wall time and the first optimum in that recorded order.


In [ ]:
method_colors = dict(zip(SEARCH_METHODS, sns.color_palette("colorblind", 4)))
for family in ensemble_spaces:
    names = [
        name
        for name, specification in ensemble_variants.items()
        if specification["family"] == family
    ]
    fig, axes = plt.subplots(
        len(names), 2, figsize=(15, 3.3 * len(names)), squeeze=False
    )
    for row_index, name in enumerate(names):
        for method in SEARCH_METHODS:
            records = ensemble_search_results[(name, method)]["records"]
            best_so_far = np.minimum.accumulate([row["mean_rmsle"] for row in records])
            unique_count = np.arange(1, len(records) + 1)
            compute_time = np.cumsum([row["compute_seconds"] for row in records])
            axes[row_index, 0].plot(
                unique_count,
                best_so_far,
                label=method,
                color=method_colors[method],
                linewidth=1.5,
            )
            axes[row_index, 1].plot(
                compute_time,
                best_so_far,
                label=method,
                color=method_colors[method],
                linewidth=1.5,
            )
        axes[row_index, 0].set(
            title=name,
            xlabel="Unique configurations evaluated",
            ylabel="Best mean CV RMSLE",
        )
        axes[row_index, 1].set(
            title=name,
            xlabel="Cumulative summed CV worker seconds",
            ylabel="Best mean CV RMSLE",
        )
        for ax in axes[row_index]:
            ax.grid(alpha=0.25)
        axes[row_index, 0].legend(fontsize=8)
    fig.suptitle(f"{family}: exhaustive search trajectories", y=1.002)
    plt.tight_layout()
    plt.show()

## Optimization methods: quality, total runtime, and discovery order

The comparison below covers all **68 completed searches**: four methods
for each of 17 model/preprocessing/encoding variants. It reads the already
recorded results, with no model training or holdout evaluation. All methods
exhaust the same 32 configurations and use identical validation folds and
seeds, so their final CV scores coincide. Error bars show the five-fold
sample standard deviation, not confidence intervals.

Three quantities answer different questions:

- **Final CV RMSLE:** the best observed mean and its fold variability.
- **Total search wall time:** the measured elapsed time for the complete
  original search, including scheduling and optimizer overhead. Cached
  results retain their original measurements.
- **First optimum in recorded order:** the number of unique configurations
  up to the first score within `1e-12` of the final optimum. Equal-quality
  configurations count as reaching the optimum even if stable tie-breaking
  selects another configuration ID. For Grid/Random, records follow
  parameter enumeration, not the completion order of parallel CV fits.

Wall time is distinct from the **summed CV worker time** in the existing
convergence plots. Grid/Random can schedule folds from several candidates
across all eight workers; TPE/GP evaluate one candidate at a time, with at
most five validation folds active. Runtime therefore reflects parallel
utilization as well as optimizer overhead. Per-candidate wall timestamps
were not recorded, so elapsed wall time to the first optimum cannot be
inferred. Each search was timed once; tiny runtime differences do not
establish an intrinsic efficiency advantage.

**Full-budget RandomizedSearchCV is a special case here.** Inspection of
scikit-learn 1.7.2 and all 17 recorded runs confirms that with list-valued
spaces and `n_iter=32` equal to the complete grid size, its candidate order
coincides with GridSearchCV's order. Sampling without replacement does
not guarantee a randomized output order. These runs therefore do not
compare a distinct random permutation with grid enumeration.
See [scikit-learn's sampling-order documentation](https://scikit-learn.org/1.7/modules/generated/sklearn.utils.random.sample_without_replacement.html).

This four-method comparison applies to all five ensemble families and their 17 variants. Part I used only GridSearchCV for Ridge, Lasso and DecisionTree; the median baseline and ordinary LinearRegression have no tuning method to compare.

In [ ]:
_optimizer_method_labels = {
    "GridSearchCV": "Grid",
    "RandomizedSearchCV": "Random",
    "Optuna TPE + completion": "TPE+completion",
    "Bayesian GP + EI": "GP+EI",
}
_optimizer_rows = []
_optimizer_order_rows = []
for _name, _specification in ensemble_variants.items():
    for _method in SEARCH_METHODS:
        _payload = ensemble_search_results[(_name, _method)]
        _records = sorted(_payload["records"], key=lambda record: record["sequence"])
        assert len(_records) == len({row["config_id"] for row in _records}) == 32
        _best = next(
            row for row in _records if row["config_id"] == _payload["best_config_id"]
        )
        _optimum = min(row["mean_rmsle"] for row in _records)
        _first_position, _first_record = next(
            (position, row)
            for position, row in enumerate(_records, start=1)
            if np.isclose(row["mean_rmsle"], _optimum, rtol=0.0, atol=1e-12)
        )
        _optimizer_rows.append(
            {
                "Model": _name,
                "Family": _specification["family"],
                "Preparation": _specification["preparation"],
                "Encoding": _specification["encoding"],
                "Method": _method,
                "Method label": _optimizer_method_labels[_method],
                "Best CV RMSLE": _best["mean_rmsle"],
                "CV fold SD": float(_best["folds"]["RMSLE"].std(ddof=1)),
                "Search wall seconds": float(_payload["search_seconds"]),
                "Unique configs to first optimum": _first_position,
                "Unique configs evaluated": len(_records),
                "Best config ID": _payload["best_config_id"],
                "First optimum config ID": _first_record["config_id"],
                "Total proposals": _payload["total_trials"],
                "Duplicate proposals": _payload["duplicate_proposals"],
                "Cache hit": _payload.get("cache_hit", False),
            }
        )
    _grid_ids = [
        row["config_id"]
        for row in ensemble_search_results[(_name, "GridSearchCV")]["records"]
    ]
    _random_ids = [
        row["config_id"]
        for row in ensemble_search_results[(_name, "RandomizedSearchCV")]["records"]
    ]
    _optimizer_order_rows.append(
        {
            "Model": _name,
            "Grid and Random use the same recorded order": _grid_ids == _random_ids,
        }
    )

optimizer_comparison = pd.DataFrame(_optimizer_rows)
optimizer_order_audit = pd.DataFrame(_optimizer_order_rows)
assert len(optimizer_comparison) == 68
assert not optimizer_comparison.duplicated(["Model", "Method"]).any()
assert optimizer_comparison["Unique configs evaluated"].eq(32).all()
assert optimizer_comparison["Unique configs to first optimum"].between(1, 32).all()
assert np.isfinite(
    optimizer_comparison[
        ["Best CV RMSLE", "CV fold SD", "Search wall seconds"]
    ].to_numpy()
).all()
for _name, _rows in optimizer_comparison.groupby("Model", sort=False):
    np.testing.assert_allclose(
        _rows["Best CV RMSLE"], _rows["Best CV RMSLE"].iloc[0], rtol=0.0, atol=1e-12
    )
    np.testing.assert_allclose(
        _rows["CV fold SD"], _rows["CV fold SD"].iloc[0], rtol=0.0, atol=1e-12
    )
print(
    "Verified 68 searches and identical final CV quality within each model variant. "
    f"Grid/Random recorded order matches in "
    f"{optimizer_order_audit.iloc[:, 1].sum()}/17 variants."
)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(
        optimizer_comparison.set_index(["Model", "Method label"])[
            [
                "Best CV RMSLE",
                "CV fold SD",
                "Search wall seconds",
                "Unique configs to first optimum",
                "Unique configs evaluated",
                "Total proposals",
                "Duplicate proposals",
            ]
        ].style.format(
            {
                "Best CV RMSLE": "{:.5f}",
                "CV fold SD": "{:.5f}",
                "Search wall seconds": "{:.2f}",
            }
        )
    )

Each row of the figures below is one fixed model variant. Colours identify
**Grid**, **Random**, **TPE+completion**, and **GP+EI** consistently across
all panels. The final-quality panel deliberately retains all four identical
results; the runtime and recorded-order panels show where their execution
differed. The dotted line at 32 marks the complete search space.

In [ ]:
from matplotlib.patches import Patch

_optimizer_labels = [_optimizer_method_labels[method] for method in SEARCH_METHODS]
_optimizer_colors = dict(zip(_optimizer_labels, sns.color_palette("colorblind", 4)))


def plot_optimizer_comparison(family):
    names = [
        name
        for name, specification in ensemble_variants.items()
        if specification["family"] == family
    ]
    figure, axes = plt.subplots(
        len(names), 3, figsize=(17.5, 3.7 * len(names)), squeeze=False
    )
    for row_index, name in enumerate(names):
        rows = (
            optimizer_comparison.loc[optimizer_comparison["Model"].eq(name)]
            .set_index("Method label")
            .loc[_optimizer_labels]
        )
        y_positions = np.arange(len(_optimizer_labels))
        colors = [_optimizer_colors[label] for label in _optimizer_labels]
        quality_axis, wall_axis, discovery_axis = axes[row_index]

        means = rows["Best CV RMSLE"].to_numpy()
        deviations = rows["CV fold SD"].to_numpy()
        span = max(
            float(np.max(means + deviations) - np.min(means - deviations)), 0.004
        )
        for position, mean, deviation, color in zip(
            y_positions, means, deviations, colors
        ):
            quality_axis.errorbar(
                mean,
                position,
                xerr=deviation,
                fmt="o",
                color=color,
                markersize=6,
                capsize=4,
                elinewidth=1.5,
                zorder=3,
            )
            quality_axis.text(
                mean + deviation + span * 0.06,
                position,
                f"{mean:.5f}",
                va="center",
                fontsize=10,
            )
        quality_axis.set_xlim(
            max(0.0, float(np.min(means - deviations)) - span * 0.10),
            float(np.max(means + deviations)) + span * 0.48,
        )
        quality_axis.set_title(f"{name}\nFinal CV quality", fontsize=12, pad=10)
        quality_axis.set_xlabel("Best mean CV RMSLE ± fold SD", fontsize=10)

        wall_seconds = rows["Search wall seconds"].to_numpy()
        wall_axis.barh(y_positions, wall_seconds, color=colors, height=0.56)
        max_wall = max(float(wall_seconds.max()), 0.1)
        for position, elapsed in zip(y_positions, wall_seconds):
            wall_axis.text(
                elapsed + max_wall * 0.025,
                position,
                f"{elapsed:.2f}s",
                va="center",
                fontsize=10,
            )
        wall_axis.set_xlim(0, max_wall * 1.29)
        wall_axis.set_title("Total measured search wall time", fontsize=12, pad=10)
        wall_axis.set_xlabel(
            "Elapsed seconds; includes scheduling + overhead", fontsize=10
        )

        discovery_positions = rows["Unique configs to first optimum"].to_numpy()
        discovery_axis.barh(y_positions, discovery_positions, color=colors, height=0.56)
        for position, count in zip(y_positions, discovery_positions):
            discovery_axis.text(
                count + 0.5, position, str(int(count)), va="center", fontsize=10
            )
        discovery_axis.axvline(32, color="#808080", linestyle=":", linewidth=1.1)
        discovery_axis.set_xlim(0, 37)
        discovery_axis.set_xticks([0, 8, 16, 24, 32])
        discovery_axis.set_title("First optimum in recorded order", fontsize=12, pad=10)
        discovery_axis.set_xlabel(
            "Unique configurations up to first optimum", fontsize=10
        )

        for axis in axes[row_index]:
            axis.set_yticks(y_positions, _optimizer_labels)
            axis.set_ylim(len(_optimizer_labels) - 0.5, -0.5)
            axis.tick_params(axis="both", labelsize=10)
            axis.grid(axis="x", alpha=0.23)
            axis.set_axisbelow(True)
        quality_axis.ticklabel_format(axis="x", style="plain", useOffset=False)

    figure.suptitle(
        f"{family}: optimization methods on the same training-CV search space",
        fontsize=17,
        y=0.994,
    )
    figure.legend(
        handles=[
            Patch(facecolor=_optimizer_colors[label], label=label)
            for label in _optimizer_labels
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.962),
        ncol=4,
        frameon=False,
        fontsize=11,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.905), h_pad=2.8, w_pad=2.0)
    plt.show()
    return figure


for _optimizer_family in ensemble_spaces:
    plot_optimizer_comparison(_optimizer_family)

# Describe the best ensemble's methods without declaring a quality winner.
_best_ensemble_name = (
    optimizer_comparison.loc[optimizer_comparison["Method"].eq("GridSearchCV")]
    .sort_values("Best CV RMSLE", kind="stable")
    .iloc[0]["Model"]
)
_best_ensemble_methods = (
    optimizer_comparison.loc[optimizer_comparison["Model"].eq(_best_ensemble_name)]
    .set_index("Method label")
    .loc[_optimizer_labels]
)
_best_mean = float(_best_ensemble_methods["Best CV RMSLE"].iloc[0])
_best_sd = float(_best_ensemble_methods["CV fold SD"].iloc[0])
_wall_description = ", ".join(
    f"{label} **{row['Search wall seconds']:.1f}s**"
    for label, row in _best_ensemble_methods.iterrows()
)
_discovery_description = ", ".join(
    f"{label} **{int(row['Unique configs to first optimum'])}**"
    for label, row in _best_ensemble_methods.iterrows()
)
display(
    Markdown(
        f"For the best ensemble variant **{_best_ensemble_name}**, all four "
        f"methods finish at **CV RMSLE {_best_mean:.5f} ± {_best_sd:.5f}**. "
        f"Total search wall times are {_wall_description}. "
        f"The optimum first appears after these unique-configuration counts "
        f"in recorded order: {_discovery_description}. "
        "GridSearchCV supplies the reference configuration for the subsequent "
        "model table; it is not a uniquely better-quality optimizer. "
        "The Grid/Random runtime difference is too small for an efficiency "
        "claim, and their recorded candidate orders coincide in this run. "
        "TPE/GP timings also include their sequential candidate scheduling. "
        "These comparisons use training CV only."
    )
)

The Ridge/Lasso alpha curves and DecisionTree depth/leaf-size heatmaps above show Part I hyperparameter selection. The median baseline and ordinary LinearRegression have no tuned hyperparameters; their preparation variants are compared in the model-comparison charts below.

## Hyperparameter selection: all tested settings and conditional profiles

Each family below has one row per preprocessing/encoding variant and one
column per tuned parameter. **Grey dots** show the mean CV RMSLE of all
32 tested configurations. Their horizontal jitter only separates overlapping
points; the parameter has exactly two tested values. **Blue lines** compare
those two values while fixing the other four parameters at the selected
configuration. **Error bars** show the sample standard deviation across
the five validation folds, and the **gold star** marks the selected value.
Lower RMSLE is better. Numeric annotations report the conditional means.

These are measured scores from the complete GridSearchCV results, not
interpolated estimates or new model fits. All four search methods have
already been checked against the same configuration-level scores. Different
conditional profiles can reflect parameter interactions: the blue line
describes the chosen context, not an isolated causal effect or a marginal
average over all other settings. Fold standard deviations are descriptive
variability, not confidence intervals. A star at a search boundary means
only that it was best within this deliberately limited space.

In [ ]:
from matplotlib.lines import Line2D

_parameter_axis_labels = {
    "n_estimators": "Number of trees",
    "iterations": "Boosting iterations",
    "max_depth": "Maximum tree depth",
    "depth": "Tree depth",
    "min_samples_leaf": "Minimum samples per leaf",
    "max_features": "Feature fraction per split",
    "max_samples": "Bootstrap sample fraction",
    "learning_rate": "Learning rate",
    "subsample": "Row subsample fraction",
    "min_child_weight": "Minimum child weight",
    "l2_leaf_reg": "L2 leaf regularization",
    "num_leaves": "Maximum leaves per tree",
    "min_child_samples": "Minimum samples per leaf",
}


def _format_parameter_level(value):
    if value is None:
        return "Unlimited"
    if isinstance(value, (float, np.floating)):
        return f"{value:g}"
    return str(value)


def plot_ensemble_parameter_selection(family):
    variant_names = [
        name
        for name, specification in ensemble_variants.items()
        if specification["family"] == family
    ]
    grid = ensemble_spaces[family]
    parameter_names = list(grid)
    figure, axes = plt.subplots(
        len(variant_names),
        len(parameter_names),
        figsize=(22, 3.8 * len(variant_names)),
        squeeze=False,
    )
    all_points_color = "#9199a2"
    profile_color = "#1668a7"
    selected_color = "#e5a000"
    jitter_rng = np.random.default_rng(SEED)

    for row_index, variant_name in enumerate(variant_names):
        payload = ensemble_search_results[(variant_name, "GridSearchCV")]
        records = payload["records"]
        selected = next(
            record
            for record in records
            if record["config_id"] == payload["best_config_id"]
        )
        assert len(records) == 32
        all_means = np.asarray([record["mean_rmsle"] for record in records])
        conditional_bounds_for_limits = []

        for column_index, parameter in enumerate(parameter_names):
            axis = axes[row_index, column_index]
            levels = grid[parameter]
            positions = np.arange(len(levels), dtype=float)
            point_positions = np.asarray(
                [levels.index(record["params"][parameter]) for record in records],
                dtype=float,
            )
            axis.scatter(
                point_positions + jitter_rng.uniform(-0.09, 0.09, len(records)),
                all_means,
                s=23,
                alpha=0.48,
                color=all_points_color,
                linewidths=0,
                zorder=1,
            )

            conditional_records = []
            for level in levels:
                matches = [
                    record
                    for record in records
                    if record["params"][parameter] == level
                    and all(
                        record["params"][other] == selected["params"][other]
                        for other in parameter_names
                        if other != parameter
                    )
                ]
                assert len(matches) == 1, (variant_name, parameter, level)
                conditional_records.append(matches[0])
            means = np.asarray([record["mean_rmsle"] for record in conditional_records])
            fold_stds = np.asarray(
                [record["folds"]["RMSLE"].std(ddof=1) for record in conditional_records]
            )
            assert np.isfinite(means).all() and np.isfinite(fold_stds).all()
            conditional_bounds_for_limits.extend(means - fold_stds)
            conditional_bounds_for_limits.extend(means + fold_stds)
            axis.errorbar(
                positions,
                means,
                yerr=fold_stds,
                color=profile_color,
                marker="o",
                markersize=5,
                linewidth=1.8,
                elinewidth=1.1,
                capsize=4,
                zorder=3,
            )
            selected_position = levels.index(selected["params"][parameter])
            axis.scatter(
                [selected_position],
                [selected["mean_rmsle"]],
                marker="*",
                s=190,
                facecolor=selected_color,
                edgecolor="#654a00",
                linewidth=0.7,
                zorder=5,
            )
            for position, mean in zip(positions, means):
                axis.annotate(
                    f"{mean:.4f}",
                    (position, mean),
                    xytext=(6 if position == 0 else -6, 7),
                    textcoords="offset points",
                    fontsize=10,
                    ha="left" if position == 0 else "right",
                    va="bottom",
                    color=profile_color,
                    bbox={
                        "facecolor": "white",
                        "alpha": 0.8,
                        "edgecolor": "none",
                        "pad": 0.4,
                    },
                    zorder=6,
                )
            short_name = parameter.removeprefix("regressor__")
            axis.set_title(
                _parameter_axis_labels.get(short_name, short_name), fontsize=12, pad=10
            )
            axis.set_xticks(
                positions, [_format_parameter_level(value) for value in levels]
            )
            axis.set_xlim(-0.28, len(levels) - 1 + 0.28)
            axis.tick_params(axis="both", labelsize=10)
            axis.grid(axis="y", alpha=0.23)
            axis.set_axisbelow(True)
            if column_index == 0:
                axis.set_ylabel(
                    f"{variant_name}\nMean CV RMSLE", fontsize=11, labelpad=9
                )

        lower = min(float(all_means.min()), min(conditional_bounds_for_limits))
        upper = max(float(all_means.max()), max(conditional_bounds_for_limits))
        padding = max((upper - lower) * 0.14, 0.002)
        for axis in axes[row_index]:
            axis.set_ylim(max(0.0, lower - padding), upper + padding)
            # The common row scale makes parameter panels comparable.
            axis.ticklabel_format(axis="y", style="plain", useOffset=False)

    legend_handles = [
        Line2D(
            [],
            [],
            linestyle="none",
            marker="o",
            color=all_points_color,
            alpha=0.6,
            markersize=5,
            label="All 32 configurations: CV means",
        ),
        Line2D(
            [],
            [],
            color=profile_color,
            marker="o",
            markersize=5,
            label="Conditional profile: other parameters fixed at winner",
        ),
        Line2D(
            [],
            [],
            linestyle="none",
            marker="|",
            color=profile_color,
            markersize=13,
            markeredgewidth=1.5,
            label="Error bars: fold SD",
        ),
        Line2D(
            [],
            [],
            linestyle="none",
            marker="*",
            color=selected_color,
            markeredgecolor="#654a00",
            markersize=12,
            label="Selected value",
        ),
    ]
    figure.suptitle(
        f"{family}: hyperparameter selection on training CV", fontsize=19, y=0.995
    )
    figure.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.969),
        ncol=2,
        frameon=False,
        fontsize=11,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.91), h_pad=2.8, w_pad=1.7)
    plt.show()
    return figure


for _parameter_family in ensemble_spaces:
    plot_ensemble_parameter_selection(_parameter_family)

## 5.6. CatBoost: native categories versus one-hot encoding

The paired comparison below holds features, folds, seeds, and the entire
32-configuration space constant. We compare each strategy's CV-selected
configuration and report its fit cost. A second, paired configuration-level
summary prevents one unusually favourable setting from representing the whole
comparison. Positive `OHE − native` RMSLE means native processing performed
better. These are descriptive CV differences after tuning, not independent
significance tests. No holdout comparison is made.

In [ ]:
catboost_encoding_rows = []
catboost_configuration_rows = []
for variant in ["base", "FE"]:
    native_name = f"CatBoost native / {variant}"
    ohe_name = f"CatBoost OHE / {variant}"
    for label, name in [("native", native_name), ("one-hot", ohe_name)]:
        catboost_encoding_rows.append(
            {
                "Preparation": variant,
                "Encoding": label,
                "CV RMSLE mean": cv_summaries[name].loc["RMSLE", "CV mean"],
                "CV RMSLE std": cv_summaries[name].loc["RMSLE", "CV std"],
                "CV MAE mean, USD": cv_summaries[name].loc["MAE", "CV mean"],
                "Mean CV fit seconds": np.mean(cv_fit_times[name]),
                "Best parameters": model_metadata[name]["parameters"],
            }
        )
    native_by_id = {
        row["config_id"]: row
        for row in ensemble_search_results[(native_name, "GridSearchCV")]["records"]
    }
    ohe_by_id = {
        row["config_id"]: row
        for row in ensemble_search_results[(ohe_name, "GridSearchCV")]["records"]
    }
    differences = np.asarray(
        [
            ohe_by_id[i]["mean_rmsle"] - native_by_id[i]["mean_rmsle"]
            for i in sorted(native_by_id)
        ]
    )
    catboost_configuration_rows.append(
        {
            "Preparation": variant,
            "Configurations compared": len(differences),
            "Mean OHE − native RMSLE": differences.mean(),
            "Median OHE − native RMSLE": np.median(differences),
            "Native lower RMSLE, configurations": int(np.sum(differences > 0)),
            "OHE lower RMSLE, configurations": int(np.sum(differences < 0)),
        }
    )

catboost_encoding_comparison = pd.DataFrame(catboost_encoding_rows)
catboost_configuration_comparison = pd.DataFrame(catboost_configuration_rows)
display(catboost_encoding_comparison.round(5))
display(catboost_configuration_comparison.round(5))

encoding_lines = []
for variant in ["base", "FE"]:
    table = catboost_encoding_comparison.query("Preparation == @variant").set_index(
        "Encoding"
    )
    difference = (
        table.loc["one-hot", "CV RMSLE mean"] - table.loc["native", "CV RMSLE mean"]
    )
    winner = (
        "native processing"
        if difference > 0
        else ("one-hot encoding" if difference < 0 else "both")
    )
    encoding_lines.append(
        f"- **{variant}:** {winner} has the lower best CV RMSLE; "
        f"OHE − native = **{difference:+.5f}**. "
        f"Mean fold fit time: native **{table.loc['native', 'Mean CV fit seconds']:.3f}s**, "
        f"one-hot **{table.loc['one-hot', 'Mean CV fit seconds']:.3f}s**."
    )
display(Markdown("\n".join(encoding_lines)))

Search/API references: [scikit-learn RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html),
[Optuna TPE](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html),
[Expected Improvement](https://scikit-optimize.readthedocs.io/en/stable/modules/generated/skopt.acquisition.gaussian_ei.html),
[CatBoost categorical processing](https://catboost.ai/docs/en/concepts/algorithm-main-stages_cat-to-numberic),
[CatBoost bootstrap](https://catboost.ai/docs/en/concepts/algorithm-main-stages_bootstrap-options),
[LightGBM parameters](https://lightgbm.readthedocs.io/en/stable/Parameters.html),
and [XGBoost feature importance](https://xgboost.readthedocs.io/en/stable/python/python_api.html).

# 6. Unified comparison and model selection

All entries below use the same five training CV splits. The selection metric is **RMSLE**; MAE and RMSE are measured in original USD, and R² also uses original prices. Each standard deviation is the sample standard deviation across five folds, not a confidence interval. Fit time includes preprocessing and model fitting for one fold; search time is the total wall time of the hyperparameter searches for that entry.

The four search methods exhaust the same finite grid. Their final optimum should therefore agree: the experiment illustrates search order and cost, rather than four independent estimates of predictive performance. Selecting a configuration on these folds makes its reported CV performance somewhat optimistic. The holdout compares all 30 fixed configurations only after the CV winner is locked; it is not used for selection.


In [ ]:
from time import perf_counter

METRIC_NAMES = ["MAE", "RMSE", "R2", "RMSLE"]

comparison_records = []
for name, folds in cv_folds.items():
    metadata = model_metadata[name]
    fit_times = np.asarray(cv_fit_times[name], dtype=float)
    assert len(folds) == len(cv_splits) == len(fit_times)
    record = {
        "Model": name,
        "Part": metadata["part"],
        "Family": metadata["family"],
        "Preparation": metadata["preparation"],
        "Categories": metadata["encoding"],
        "Search method": metadata["method"],
        "CV fit seconds mean": fit_times.mean(),
        "CV fit seconds std": fit_times.std(ddof=1),
        "Search seconds": float(search_times.get(name, 0.0)),
        "Parameters": metadata["parameters"],
    }
    for metric in METRIC_NAMES:
        record[f"{metric} mean"] = folds[metric].mean()
        record[f"{metric} std"] = folds[metric].std(ddof=1)
    comparison_records.append(record)

leaderboard = (
    pd.DataFrame(comparison_records)
    .sort_values(["RMSLE mean", "Model"], kind="stable")
    .set_index("Model")
)
leaderboard_display = leaderboard[
    ["Part", "Family", "Preparation", "Categories", "Search method"]
].copy()
for metric in METRIC_NAMES:
    precision = 4 if metric in ["R2", "RMSLE"] else 2
    leaderboard_display[f"CV {metric} mean ± std"] = [
        f"{mean:,.{precision}f} ± {std:,.{precision}f}"
        for mean, std in zip(
            leaderboard[f"{metric} mean"], leaderboard[f"{metric} std"]
        )
    ]
leaderboard_display["CV fit seconds mean ± std"] = [
    f"{mean:.3f} ± {std:.3f}"
    for mean, std in zip(
        leaderboard["CV fit seconds mean"], leaderboard["CV fit seconds std"]
    )
]
leaderboard_display["Total search seconds"] = leaderboard["Search seconds"].round(2)
leaderboard_display["Key parameters"] = leaderboard["Parameters"]
display(leaderboard_display)

# One compact row per family; the complete table above retains every preparation.
trained_ranking = leaderboard.loc[
    [name for name in leaderboard.index if name in model_candidates]
]
family_winners = (
    trained_ranking.reset_index().drop_duplicates(["Part", "Family"]).set_index("Model")
)
display(
    family_winners[
        [
            "Part",
            "Family",
            "Preparation",
            "Categories",
            "RMSLE mean",
            "RMSLE std",
            "MAE mean",
            "RMSE mean",
            "R2 mean",
            "CV fit seconds mean",
            "Search seconds",
            "Parameters",
        ]
    ].round(4)
)

# Keep the native-versus-one-hot comparison visible even if another family wins.
catboost_rows = leaderboard[leaderboard["Family"].eq("CatBoost")]
if not catboost_rows.empty:
    display(
        catboost_rows[
            [
                "Preparation",
                "Categories",
                "RMSLE mean",
                "RMSLE std",
                "MAE mean",
                "CV fit seconds mean",
                "Parameters",
            ]
        ]
        .sort_values(["Preparation", "Categories"])
        .round(4)
    )
    catboost_comparison_lines = []
    for preparation in ["base", "FE"]:
        native_names = catboost_rows[
            catboost_rows["Preparation"].eq(preparation)
            & catboost_rows["Categories"].eq("native")
        ].index.tolist()
        onehot_names = catboost_rows[
            catboost_rows["Preparation"].eq(preparation)
            & catboost_rows["Categories"].eq("one-hot")
        ].index.tolist()
        if native_names and onehot_names:
            native_name, onehot_name = native_names[0], onehot_names[0]
            native_gain = (
                cv_folds[onehot_name]["RMSLE"].to_numpy()
                - cv_folds[native_name]["RMSLE"].to_numpy()
            )
            native_fit_ratio = (
                leaderboard.loc[native_name, "CV fit seconds mean"]
                / leaderboard.loc[onehot_name, "CV fit seconds mean"]
            )
            catboost_comparison_lines.append(
                f"- **{preparation}:** native-category CV RMSLE "
                f"**{leaderboard.loc[native_name, 'RMSLE mean']:.4f}** vs one-hot "
                f"**{leaderboard.loc[onehot_name, 'RMSLE mean']:.4f}**. "
                f"Paired improvement from native handling is "
                f"**{native_gain.mean():+.5f} ± {native_gain.std(ddof=1):.5f}**, "
                f"with lower RMSLE in **{int((native_gain > 0).sum())}/5** folds; "
                f"the native/one-hot mean fit-time ratio is **{native_fit_ratio:.2f}×**."
            )
    catboost_comparison_lines.extend(
        [
            "",
            "Both category representations use the same numerical preprocessing, folds, "
            "and finite hyperparameter grid. These comparisons concern the best tuned "
            "configuration of each representation; their selected parameters can differ. "
            "The numeric-only high-correlation branch has no categorical encoding to compare.",
        ]
    )
    display(Markdown("\n".join(catboost_comparison_lines)))

## 6.1. Lock the final configuration and assess the tradeoff

Choose the smallest mean training-CV RMSLE across all trained candidates; use the model name as a deterministic tie-breaker. The following comparison uses paired fold scores because every model sees the same splits. It is descriptive: the folds have already been used for tuning, so neither their spread nor the count of improved folds establishes statistical significance.


In [ ]:
best_model_name = min(
    model_candidates,
    key=lambda name: (cv_folds[name]["RMSLE"].mean(), name),
)
best_part_i_name = min(
    (name for name in model_candidates if model_metadata[name]["part"] == "I"),
    key=lambda name: (cv_folds[name]["RMSLE"].mean(), name),
)
best_part_ii_name = min(
    (name for name in model_candidates if model_metadata[name]["part"] == "II"),
    key=lambda name: (cv_folds[name]["RMSLE"].mean(), name),
)
paired_improvement = (
    cv_folds[best_part_i_name]["RMSLE"].to_numpy()
    - cv_folds[best_part_ii_name]["RMSLE"].to_numpy()
)
relative_cv_gain = (
    100 * paired_improvement.mean() / cv_folds[best_part_i_name]["RMSLE"].mean()
)
part_i_fit_seconds = leaderboard.loc[best_part_i_name, "CV fit seconds mean"]
part_ii_fit_seconds = leaderboard.loc[best_part_ii_name, "CV fit seconds mean"]
fit_time_ratio = part_ii_fit_seconds / part_i_fit_seconds
part_i_mae = leaderboard.loc[best_part_i_name, "MAE mean"]
part_ii_mae = leaderboard.loc[best_part_ii_name, "MAE mean"]
tradeoff_lines = [
    f"**Final configuration locked before the holdout: {best_model_name}.**",
    "",
    f"The best Part I model is **{best_part_i_name}** "
    f"(CV RMSLE {leaderboard.loc[best_part_i_name, 'RMSLE mean']:.4f}); "
    f"the best Part II model is **{best_part_ii_name}** "
    f"({leaderboard.loc[best_part_ii_name, 'RMSLE mean']:.4f}).",
    "",
    f"Part II changes mean RMSLE by **{relative_cv_gain:+.2f}% improvement**. "
    f"The paired improvement (Part I minus Part II) is "
    f"**{paired_improvement.mean():+.5f} ± {paired_improvement.std(ddof=1):.5f}** "
    f"across folds, with lower RMSLE in **{int((paired_improvement > 0).sum())}/5** folds. "
    f"Mean dollar MAE changes from **${part_i_mae:,.2f}** to **${part_ii_mae:,.2f}**.",
    "",
    f"One CV fit takes **{part_i_fit_seconds:.3f} s** for the Part I winner and "
    f"**{part_ii_fit_seconds:.3f} s** for the Part II winner "
    f"(**{fit_time_ratio:.2f}×**). The Part II winner's four searches take "
    f"**{search_times[best_part_ii_name]:,.1f} s** in total. "
    "Search cost is a development cost; the fit time is closer to the cost of retraining.",
    "",
]
if model_metadata[best_model_name]["part"] == "II":
    if (
        relative_cv_gain < 3
        or paired_improvement.mean() <= paired_improvement.std(ddof=1)
        or (paired_improvement > 0).sum() < 4
    ):
        caution_reasons = []
        if relative_cv_gain < 3:
            caution_reasons.append(
                f"the relative mean RMSLE reduction is only {relative_cv_gain:.2f}%"
            )
        if paired_improvement.mean() <= paired_improvement.std(ddof=1):
            caution_reasons.append(
                "the mean paired gain is no larger than its fold-to-fold spread"
            )
        if (paired_improvement > 0).sum() < 4:
            caution_reasons.append(
                f"only {int((paired_improvement > 0).sum())}/5 folds improve"
            )
        tradeoff_lines.append(
            "The Part II model remains the benchmark winner under the fixed minimum-CV-RMSLE "
            f"rule. However, {'; '.join(caution_reasons)}, so the observed advantage does not "
            "automatically justify extra tuning, dependencies, and retraining cost for routine "
            "use. The benchmark values accuracy first and has no fixed runtime budget; a "
            "deployment decision would also have to value the measured cost ratio. These "
            "descriptive comparisons are not significance tests or universal acceptance thresholds."
        )
    else:
        tradeoff_lines.append(
            "For this accuracy-first benchmark, the observed RMSLE gain supports the added "
            "model complexity: the mean paired improvement is larger than its descriptive "
            "fold-to-fold spread. The measured fit-time ratio and search cost above quantify "
            "the price of that improvement; extra tuning and boosting dependencies also "
            "increase maintenance work. This is a practical assessment for this experiment, "
            "not a claim of statistical significance or a universal operational advantage."
        )
else:
    tradeoff_lines.append(
        "The added ensemble complexity is not justified by a lower observed CV RMSLE "
        "in this experiment: a Part I configuration remains the selected model. "
        "The best ensemble still provides a concrete comparison of quality and cost."
    )
display(Markdown("\n".join(tradeoff_lines)))

# This is the only fit of the selected configuration on the complete training split.
final_model = clone(model_candidates[best_model_name])
final_fit_started = perf_counter()
final_model.fit(X, y)
final_fit_seconds = perf_counter() - final_fit_started

### Decision for this experiment

For this accuracy-first study, the additional model complexity is justified: CatBoost OHE/FE improves mean CV RMSLE by **6.41%**, improves all five paired folds, and takes only **0.714 seconds** to fit the full training split on this machine. Its roughly **7.62×** fold-fit cost is small in absolute terms. The extensive four-method benchmark is a development exercise; using the selected configuration does not require repeating every search.

The fold-to-fold variation of the paired gain is substantial, so these results support the notebook's fixed CV selection rule rather than a claim of statistical significance or a universal production advantage. Lasso remains a useful simpler reference. This judgement uses training/CV results and training time; the terminal holdout is reserved for the fixed-configuration reporting comparison.


## Training comparison: all candidates and selected family representatives

The first chart compares all 30 entries on the common training-CV splits. The remaining charts compare the baseline and the best CV configuration from each model family, including the selected preparation and categorical representation. These representatives are fixed from the existing CV ranking; the graphs do not change the final winner.

Refit every one of the 30 fixed configurations on the same five folds and record training and validation scores; retain one CV-selected representative per family in the compact charts. Quality error bars show the sample standard deviation across folds, **not confidence intervals**. An optimistic training score and a large validation gap can indicate overfitting; both also depend on model capacity and the finite sample. The validation scores have already participated in parameter selection and are not independent estimates of the tuning procedure's performance.


In [ ]:
from time import perf_counter
from sklearn.model_selection import cross_validate

comparison_baseline_names = [name for name in cv_folds if name not in model_candidates]
assert len(comparison_baseline_names) == 1, (
    "Expected one registered median-price baseline."
)
comparison_baseline_name = comparison_baseline_names[0]

# Lock a single representative per family, plus the original-price baseline.
comparison_model_names = sorted(
    set(family_winners.index).union({comparison_baseline_name}),
    key=lambda name: (cv_folds[name]["RMSLE"].mean(), name),
)
training_comparison_all_names = sorted(
    cv_folds,
    key=lambda name: (cv_folds[name]["RMSLE"].mean(), name),
)
assert best_model_name in comparison_model_names
comparison_baseline_scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "RMSLE": "neg_root_mean_squared_log_error",
}
training_comparison_folds = {}
training_comparison_records = []

for name in training_comparison_all_names:
    is_baseline = name == comparison_baseline_name
    estimator = clone(baseline if is_baseline else model_candidates[name])
    target = y_price if is_baseline else y
    fold_scores = cross_validate(
        estimator,
        X,
        target,
        cv=cv_splits,
        scoring=comparison_baseline_scoring if is_baseline else scoring,
        return_train_score=True,
        return_estimator=True,
        n_jobs=CV_JOBS,
        error_score="raise",
    )
    comparison_folds = pd.DataFrame(index=pd.Index(range(1, 6), name="Fold"))
    for metric in ["MAE", "RMSE", "R2", "RMSLE"]:
        sign = 1 if metric == "R2" else -1
        comparison_folds[f"Train {metric}"] = sign * fold_scores[f"train_{metric}"]
        comparison_folds[f"Validation {metric}"] = sign * fold_scores[f"test_{metric}"]
        np.testing.assert_allclose(
            comparison_folds[f"Validation {metric}"].to_numpy(),
            cv_folds[name][metric].to_numpy(),
            rtol=1e-5,
            atol=1e-6,
            err_msg=f"The fixed configuration must reproduce registered CV scores: {name}, {metric}",
        )
    comparison_folds["Fit seconds"] = fold_scores["fit_time"]
    train_prediction_seconds, validation_prediction_seconds = [], []
    for fitted, (train_positions, validation_positions) in zip(
        fold_scores["estimator"], cv_splits
    ):
        prediction_started = perf_counter()
        train_prediction = fitted.predict(X.iloc[train_positions])
        train_prediction_seconds.append(perf_counter() - prediction_started)
        prediction_started = perf_counter()
        validation_prediction = fitted.predict(X.iloc[validation_positions])
        validation_prediction_seconds.append(perf_counter() - prediction_started)
        assert (
            np.isfinite(train_prediction).all()
            and np.isfinite(validation_prediction).all()
        )
    comparison_folds["Train prediction seconds"] = train_prediction_seconds
    comparison_folds["Validation prediction seconds"] = validation_prediction_seconds
    comparison_folds["Training rows"] = [
        len(train_positions) for train_positions, _ in cv_splits
    ]
    comparison_folds["Validation rows"] = [
        len(validation_positions) for _, validation_positions in cv_splits
    ]
    comparison_folds["RMSLE generalization gap"] = (
        comparison_folds["Validation RMSLE"] - comparison_folds["Train RMSLE"]
    )
    assert np.isfinite(comparison_folds.to_numpy()).all()
    training_comparison_folds[name] = comparison_folds
    row = {"Model": name, "Family": model_metadata[name]["family"]}
    for column in comparison_folds.columns:
        row[f"{column} mean"] = comparison_folds[column].mean()
        row[f"{column} std"] = comparison_folds[column].std(ddof=1)
    training_comparison_records.append(row)
    del fold_scores

training_comparison_summary = (
    pd.DataFrame(training_comparison_records)
    .set_index("Model")
    .loc[training_comparison_all_names]
)
training_plot_summary = training_comparison_summary.loc[comparison_model_names]
training_comparison_table = training_comparison_summary[["Family"]].copy()
for metric in ["MAE", "RMSE", "R2", "RMSLE"]:
    precision = 4 if metric in ["R2", "RMSLE"] else 2
    for split in ["Train", "Validation"]:
        training_comparison_table[f"{split} {metric} mean ± std"] = [
            f"{mean:,.{precision}f} ± {std:,.{precision}f}"
            for mean, std in zip(
                training_comparison_summary[f"{split} {metric} mean"],
                training_comparison_summary[f"{split} {metric} std"],
            )
        ]
for timing in [
    "Fit seconds",
    "Train prediction seconds",
    "Validation prediction seconds",
]:
    training_comparison_table[f"{timing} mean ± std"] = [
        f"{mean:.4f} ± {std:.4f}"
        for mean, std in zip(
            training_comparison_summary[f"{timing} mean"],
            training_comparison_summary[f"{timing} std"],
        )
    ]
display(training_comparison_table)

### All 30 baseline and model entries on training CV

Order follows mean CV RMSLE, with model names breaking ties. Colors identify the study part, and the selected final configuration is highlighted. The plot includes every preparation and both CatBoost category representations wherever applicable.


In [ ]:
all_cv_plot_names = sorted(
    cv_folds,
    key=lambda name: (cv_folds[name]["RMSLE"].mean(), name),
)
all_cv_rmsle_mean = np.asarray(
    [cv_folds[name]["RMSLE"].mean() for name in all_cv_plot_names]
)
all_cv_rmsle_std = np.asarray(
    [cv_folds[name]["RMSLE"].std(ddof=1) for name in all_cv_plot_names]
)
all_cv_colors = [
    "crimson"
    if name == best_model_name
    else (
        "slategray"
        if name == comparison_baseline_name
        else ("steelblue" if model_metadata[name]["part"] == "I" else "darkorange")
    )
    for name in all_cv_plot_names
]
positions = np.arange(len(all_cv_plot_names))
fig, ax = plt.subplots(figsize=(12, max(8, len(all_cv_plot_names) * 0.32)))
ax.barh(
    positions,
    all_cv_rmsle_mean,
    xerr=all_cv_rmsle_std,
    color=all_cv_colors,
    alpha=0.85,
    capsize=2,
)
ax.set_yticks(positions, labels=all_cv_plot_names)
ax.invert_yaxis()
for label, name in zip(ax.get_yticklabels(), all_cv_plot_names):
    if name == best_model_name:
        label.set_color("crimson")
        label.set_fontweight("bold")
ax.set(
    xlabel="Validation RMSLE (mean ± sample fold std; lower is better)",
    title="All 30 entries on the common five training-CV folds",
)
ax.grid(axis="x", alpha=0.2)
ax.legend(
    handles=[
        Patch(facecolor="slategray", label="Median-price baseline"),
        Patch(facecolor="steelblue", label="Part I"),
        Patch(facecolor="darkorange", label="Part II"),
        Patch(facecolor="crimson", label="Selected final configuration"),
    ],
    loc="upper right",
)
plt.tight_layout()
plt.show()

### Training versus validation quality for each family representative

The same row order is used for all four metrics. Lower MAE, RMSE, and RMSLE are better; higher R² is better. The baseline is retained on the shared axes to show the size of the improvement over a constant predictor. Red, bold labels identify the already selected final configuration.


In [ ]:
positions = np.arange(len(comparison_model_names))
fig, axes = plt.subplots(2, 2, figsize=(19, max(9, len(comparison_model_names) * 0.85)))
metric_labels = {
    "MAE": "MAE, USD (lower is better)",
    "RMSE": "RMSE, USD (lower is better)",
    "R2": "R² (higher is better)",
    "RMSLE": "RMSLE (lower is better)",
}
for metric, ax in zip(["MAE", "RMSE", "R2", "RMSLE"], axes.flat):
    for split, offset, color in [
        ("Train", -0.18, "steelblue"),
        ("Validation", 0.18, "darkorange"),
    ]:
        ax.barh(
            positions + offset,
            training_plot_summary[f"{split} {metric} mean"],
            height=0.32,
            xerr=training_plot_summary[f"{split} {metric} std"],
            color=color,
            alpha=0.8,
            capsize=2,
            label=split,
        )
    ax.set_yticks(positions, labels=comparison_model_names)
    ax.invert_yaxis()
    for label, name in zip(ax.get_yticklabels(), comparison_model_names):
        if name == best_model_name:
            label.set_color("crimson")
            label.set_fontweight("bold")
    ax.set(
        xlabel=metric_labels[metric], title=f"{metric}: fixed family representatives"
    )
    ax.axvline(0, color="black", linewidth=0.6)
    ax.grid(axis="x", alpha=0.2)
    ax.legend()
fig.suptitle("Training and validation means ± sample fold standard deviation", y=1.01)
plt.tight_layout()
plt.show()

### Generalization gap and fitting cost

The paired RMSLE gap is validation minus training error on each fold. Positive values indicate an optimistic training error. The fit-time chart uses a logarithmic scale so that a fast baseline and more expensive tree ensembles remain visible; points show the mean and whiskers show the observed minimum and maximum across folds. Times include the selected preprocessing and are measured on this machine during these fixed-configuration refits; concurrent folds can affect wall-clock timings.

Prediction timings in the table measure one extra call to `predict` on a training or validation fold, including the pipeline transformations. They are descriptive measurements, not isolated latency benchmarks; no additional hyperparameters are searched for these charts.


In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(18, max(5, len(comparison_model_names) * 0.48)), sharey=True
)
positions = np.arange(len(comparison_model_names))
representative_colors = [
    "crimson" if name == best_model_name else "steelblue"
    for name in comparison_model_names
]
axes[0].barh(
    positions,
    training_plot_summary["RMSLE generalization gap mean"],
    xerr=training_plot_summary["RMSLE generalization gap std"],
    color=representative_colors,
    capsize=2,
    alpha=0.85,
)
axes[0].axvline(0, color="black", linestyle="--", linewidth=0.8)
axes[0].set(
    xlabel="Validation − train RMSLE (mean ± paired fold std)",
    title="Generalization gap of fixed configurations",
)

# The observed range is strictly positive and remains meaningful on a log axis.
fit_means = training_plot_summary["Fit seconds mean"].to_numpy()
fit_minima = np.asarray(
    [
        training_comparison_folds[name]["Fit seconds"].min()
        for name in comparison_model_names
    ]
)
fit_maxima = np.asarray(
    [
        training_comparison_folds[name]["Fit seconds"].max()
        for name in comparison_model_names
    ]
)
axes[1].errorbar(
    fit_means,
    positions,
    xerr=np.vstack([fit_means - fit_minima, fit_maxima - fit_means]),
    fmt="none",
    ecolor="gray",
    capsize=2,
    zorder=1,
)
axes[1].scatter(fit_means, positions, color=representative_colors, s=55, zorder=2)
axes[1].set_xscale("log")
axes[1].set(
    xlabel="One CV fit, seconds (log scale; mean and observed fold range)",
    title="Fitting cost, including preprocessing",
)
axes[0].set_yticks(positions, labels=comparison_model_names)
axes[0].invert_yaxis()
for label, name in zip(axes[0].get_yticklabels(), comparison_model_names):
    if name == best_model_name:
        label.set_color("crimson")
        label.set_fontweight("bold")
for ax in axes:
    ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

assert (
    len(training_comparison_folds)
    == len(training_comparison_all_names)
    == len(cv_folds)
)
assert best_model_name in training_comparison_summary.index
assert all(len(folds) == 5 for folds in training_comparison_folds.values())
assert np.isfinite(
    training_comparison_summary.select_dtypes(include="number").to_numpy()
).all()
print(
    f"Refitted all {len(cv_folds)} fixed configurations; "
    f"compact charts show {len(comparison_model_names)} family representatives including the baseline."
)

# 7. Explain the final configuration using training CV

## 7.1. Permutation importance on validation folds

Refit the locked final configuration on each training fold and permute **original input columns** on its validation fold. The full pipeline recomputes engineered features after permutation, so a raw column's importance includes its influence through derived features. Importance is the **increase in validation RMSLE** after shuffling: positive values indicate useful predictive information; negative values can occur through sampling variation.

There are ten shuffles per feature and fold with a fixed seed. The table and error bars show the mean and sample standard deviation of the five fold-level means; the within-fold shuffle standard deviation is reported separately. Correlated or substitutable predictors can share information, so a low individual importance does not imply that a feature is intrinsically uninformative. These are CV diagnostics of the selected configuration, on folds also used for selection; no holdout values are used.


In [ ]:
final_fold_models = []
final_oof_log_predictions = np.full(len(X), np.nan)
permutation_by_fold = []
for fold_number, (train_positions, validation_positions) in enumerate(cv_splits):
    fold_model = clone(model_candidates[best_model_name])
    fold_model.fit(X.iloc[train_positions], y.iloc[train_positions])
    final_oof_log_predictions[validation_positions] = fold_model.predict(
        X.iloc[validation_positions]
    )
    fold_prices = inverse_predictions(final_oof_log_predictions[validation_positions])
    observed_fold_metrics = np.asarray(
        [
            mean_absolute_error(y_price.iloc[validation_positions], fold_prices),
            root_mean_squared_error(y_price.iloc[validation_positions], fold_prices),
            r2_score(y_price.iloc[validation_positions], fold_prices),
            root_mean_squared_log_error(
                y_price.iloc[validation_positions], fold_prices
            ),
        ]
    )
    np.testing.assert_allclose(
        observed_fold_metrics,
        cv_folds[best_model_name].iloc[fold_number][METRIC_NAMES].to_numpy(dtype=float),
        rtol=1e-5,
        atol=1e-6,
        err_msg="Refitted final configuration must reproduce the registered fold metrics.",
    )
    fold_importance = permutation_importance(
        fold_model,
        X.iloc[validation_positions],
        y.iloc[validation_positions],
        scoring=scoring["RMSLE"],
        n_repeats=10,
        random_state=SEED,
        n_jobs=CV_JOBS,
    )
    final_fold_models.append(fold_model)
    permutation_by_fold.append(fold_importance.importances)

assert np.isfinite(final_oof_log_predictions).all()
final_oof_predictions = inverse_predictions(final_oof_log_predictions)
permutation_array = np.stack(permutation_by_fold)
permutation_fold_means = permutation_array.mean(axis=2)
permutation_table = pd.DataFrame(
    {
        "RMSLE increase mean": permutation_fold_means.mean(axis=0),
        "RMSLE increase CV std": permutation_fold_means.std(axis=0, ddof=1),
        "Within-fold shuffle std mean": permutation_array.std(axis=2, ddof=1).mean(
            axis=0
        ),
    },
    index=pd.Index(X.columns, name="Original feature"),
).sort_values("RMSLE increase mean", ascending=False, kind="stable")
display(permutation_table.head(25).round(5))

plot_importance = permutation_table.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    plot_importance.index,
    plot_importance["RMSLE increase mean"],
    xerr=plot_importance["RMSLE increase CV std"],
    color="steelblue",
    alpha=0.85,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(
    xlabel="Increase in validation RMSLE after shuffling (mean ± fold std)",
    title=f"CV permutation importance: {best_model_name}",
)
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 7.2. Compare with built-in tree importance

For one-hot pipelines, sum the importance of every category dummy back to its pre-encoding column. Missing-value indicators also belong to their source numeric column. The mapping below uses fitted transformer metadata, rather than splitting feature names on underscores. Individual dummy rankings can fragment one categorical feature's importance and overemphasize a single frequent category; they are not a fair ranking of original columns.

Engineered features retain their own names. Built-in importance allocates credit between original and engineered features, whereas raw-column permutation also changes all dependent engineered features. Their rankings therefore need not agree. Tree impurity/gain measures can favor features with many candidate splits and correlated predictors can divide credit. These measures describe model use, not causal effects. The measures also have different definitions: Random Forest and Gradient Boosting use impurity decrease, XGBoost uses total gain, LightGBM uses gain, and CatBoost uses PredictionValuesChange. Only ranks and normalized shares are compared, not the numerical scales of different definitions.


In [ ]:
def transformed_source_columns(fitted_preparation):
    """Return one pre-encoding parent per transformed model column."""
    fitted_preprocessor = fitted_preparation.named_steps["preprocessor"]
    if hasattr(fitted_preprocessor, "source_columns_"):
        sources = list(fitted_preprocessor.source_columns_)
        assert len(sources) == len(fitted_preprocessor.output_columns_)
        return sources

    sources = []
    for block_name, transformer, columns in fitted_preprocessor.transformers_:
        if isinstance(transformer, str) and transformer == "drop":
            continue
        columns = list(columns)
        if block_name == "numeric":
            fitted_imputer = transformer.named_steps["imputer"]
            parent_by_name = {column: column for column in columns}
            parent_by_name.update(
                {f"missingindicator_{column}": column for column in columns}
            )
            output_names = fitted_imputer.get_feature_names_out(columns)
            sources.extend(parent_by_name[output_name] for output_name in output_names)
        elif block_name == "categorical":
            encoder = transformer.named_steps["onehot"]
            try:
                infrequent = encoder.infrequent_categories_
            except AttributeError:
                infrequent = [None] * len(columns)
            for position, column in enumerate(columns):
                output_count = len(encoder.categories_[position])
                if infrequent[position] is not None:
                    output_count -= len(infrequent[position]) - 1
                if (
                    encoder.drop_idx_ is not None
                    and encoder.drop_idx_[position] is not None
                ):
                    output_count -= 1
                sources.extend([column] * output_count)
        else:
            raise AssertionError(f"Unexpected transformed block: {block_name}")
    return sources


def grouped_tree_importance(fitted_pipeline):
    regressor = fitted_pipeline.named_steps["regressor"]
    if not hasattr(regressor, "feature_importances_"):
        return None
    values = np.asarray(regressor.feature_importances_, dtype=float)
    sources = transformed_source_columns(fitted_pipeline.named_steps["preparation"])
    assert len(values) == len(sources), (
        "Every transformed feature must map to exactly one parent."
    )
    assert np.isfinite(values).all() and (values >= 0).all()
    grouped = (
        pd.Series(values, index=pd.Index(sources, name="Pre-encoding feature"))
        .groupby(level=0, sort=False)
        .sum()
    )
    return grouped / grouped.sum() if grouped.sum() > 0 else grouped


native_importance = grouped_tree_importance(final_model)
importance_comparison = None
if native_importance is None:
    display(
        Markdown(
            f"**{best_model_name}** has no built-in tree feature importance. "
            "Permutation importance above remains available; a tree model is not substituted "
            "after selection just to obtain a built-in ranking."
        )
    )
else:
    native_folds = pd.concat(
        [
            grouped_tree_importance(fold_model).rename(f"Fold {i + 1}")
            for i, fold_model in enumerate(final_fold_models)
        ],
        axis=1,
    ).fillna(0.0)
    native_table = native_importance.rename(
        "Full-train normalized built-in importance"
    ).to_frame()
    native_table["CV built-in importance mean"] = native_folds.mean(axis=1)
    native_table["CV built-in importance std"] = native_folds.std(axis=1, ddof=1)
    native_table = native_table.sort_values(
        "Full-train normalized built-in importance", ascending=False, kind="stable"
    )
    display(native_table.head(25).round(5))
    importance_comparison = permutation_table.join(native_table, how="outer")
    importance_comparison["Permutation rank"] = importance_comparison[
        "RMSLE increase mean"
    ].rank(ascending=False, method="min")
    importance_comparison["Built-in rank"] = importance_comparison[
        "Full-train normalized built-in importance"
    ].rank(ascending=False, method="min")
    display(importance_comparison.sort_values("Permutation rank").head(25).round(5))
    common_features = permutation_table.index.intersection(native_importance.index)
    if len(common_features) >= 3:
        common_rank_correlation = permutation_table.loc[
            common_features, "RMSLE increase mean"
        ].corr(native_importance.loc[common_features], method="spearman")
        display(
            Markdown(
                f"Spearman agreement across **{len(common_features)} shared raw feature names**: "
                f"**{common_rank_correlation:.3f}**. Engineered features have separate built-in "
                "rows and are excluded from this rank comparison. Agreement is descriptive; "
                "the two methods allocate credit at different levels."
            )
        )
    native_plot = native_table.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(
        native_plot.index,
        native_plot["Full-train normalized built-in importance"],
        color="darkorange",
    )
    ax.set(
        xlabel="Normalized built-in importance, summed by pre-encoding column",
        title=f"Built-in feature importance: {best_model_name}",
    )
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.show()

### What the two importance rankings show here

Shuffling **GrLivArea** increases validation RMSLE by **0.0838** on average, followed by **OverallQual (0.0466)** and **TotalBsmtSF (0.0248)**. Built-in CatBoost importance instead assigns its largest normalized shares to the engineered **QualityArea (15.79%)**, **TotalSF (11.39%)**, and **LogTotalSF (6.56%)**, alongside **OverallQual (9.87%)**.

This difference is expected: `QualityArea = GrLivArea × OverallQual` and `TotalSF = GrLivArea + TotalBsmtSF`. Permuting an original area or quality column changes these derived inputs together, while built-in importance divides credit among their separate model columns. Thus GrLivArea ranks first by permutation but only twelfth by the full-train built-in measure. Across 50 shared raw feature names, rank agreement is nevertheless fairly strong (**Spearman 0.810**). These rankings describe predictive dependence, not causal price effects.


## 7.3. Compare preprocessing branches with out-of-fold residuals

The best model in each preparation branch was fixed by CV RMSLE. Every residual below uses a prediction from a model trained without that row. The final configuration's fold predictions are reused where applicable. These descriptive train-CV plots replace the previous multi-model holdout comparison and do not change the locked winner. Positive residuals indicate underprediction.


In [ ]:
variant_winners = {
    preparation: min(
        (
            name
            for name in model_candidates
            if model_metadata[name]["preparation"] == preparation
        ),
        key=lambda name: (cv_folds[name]["RMSLE"].mean(), name),
    )
    for preparation in ["base", "FE", "high_corr"]
}
oof_predictions_by_model = {best_model_name: final_oof_predictions}
oof_predictions = pd.DataFrame(index=X.index)
oof_labels = {}
for preparation, winner_name in variant_winners.items():
    if winner_name not in oof_predictions_by_model:
        predictions_log = np.full(len(X), np.nan)
        for train_positions, validation_positions in cv_splits:
            branch_model = clone(model_candidates[winner_name])
            branch_model.fit(X.iloc[train_positions], y.iloc[train_positions])
            predictions_log[validation_positions] = branch_model.predict(
                X.iloc[validation_positions]
            )
        assert np.isfinite(predictions_log).all()
        oof_predictions_by_model[winner_name] = inverse_predictions(predictions_log)
    oof_predictions[preparation] = oof_predictions_by_model[winner_name]
    oof_labels[preparation] = f"{preparation}: {winner_name}"

oof_residuals = oof_predictions.apply(lambda prediction: y_price - prediction)
oof_residual_summary = pd.DataFrame(
    {
        "CV-selected branch winner": pd.Series(variant_winners),
        "OOF mean residual, USD": oof_residuals.mean(),
        "OOF MAE, USD": oof_residuals.abs().mean(),
        "OOF RMSE, USD": np.sqrt(oof_residuals.pow(2).mean()),
    }
)
display(oof_residual_summary.round(2))

residual_bound = max(float(oof_residuals.abs().to_numpy().max()), 1.0)
common_bins = np.linspace(-residual_bound, residual_bound, 41)
palette = dict(zip(oof_residuals.columns, sns.color_palette("colorblind", 3)))
fig, axes = plt.subplots(
    2, 1, figsize=(12, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)
for preparation in oof_residuals:
    axes[0].hist(
        oof_residuals[preparation],
        bins=common_bins,
        density=True,
        histtype="step",
        linewidth=1.8,
        color=palette[preparation],
        label=oof_labels[preparation],
    )
axes[0].set(title="Training OOF residual distributions", ylabel="Density")
axes[0].legend(title="Branch winners selected by training CV", fontsize=8)
sns.boxplot(
    data=oof_residuals, orient="h", palette=palette, showfliers=True, ax=axes[1]
)
axes[1].set(
    title="OOF residual spread and outliers", xlabel="Actual − predicted price, USD"
)
for ax in axes:
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlim(-residual_bound * 1.03, residual_bound * 1.03)
    ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

# 8. Holdout reporting after CV selection

The final configuration, preprocessing, and interpretation procedure are now fixed. Predict the selected model once and store its array for final-model diagnostics. Then compare all 30 fixed configurations in a separate report-only pass; the selected array is reused. No model is selected or retuned from holdout results.

**Historical disclosure:** this repository already contained results for this same holdout split. It cannot be described as a previously unseen external dataset. The previous revision evaluated only the CV winner. At the user's subsequent request, this revision expands reporting to all 30 configurations, with one prediction per configuration in the reporting pass.


In [ ]:
if globals().get("holdout_evaluation_count", 0) != 0:
    raise RuntimeError(
        "The holdout has already been evaluated. Reuse y_pred_holdout or restart the kernel."
    )

X_holdout = test.drop(columns=["SalePrice"])
y_holdout = test["SalePrice"].copy()
assert X.index.intersection(X_holdout.index).empty
holdout_prediction_started = perf_counter()
y_pred_holdout = inverse_predictions(final_model.predict(X_holdout))
holdout_prediction_seconds = perf_counter() - holdout_prediction_started
holdout_evaluation_count = 1
assert len(y_pred_holdout) == len(y_holdout) and np.isfinite(y_pred_holdout).all()
holdout_metrics = pd.Series(
    {
        "MAE": mean_absolute_error(y_holdout, y_pred_holdout),
        "RMSE": root_mean_squared_error(y_holdout, y_pred_holdout),
        "R2": r2_score(y_holdout, y_pred_holdout),
        "RMSLE": root_mean_squared_log_error(y_holdout, y_pred_holdout),
    },
    name=best_model_name,
)
display(holdout_metrics.to_frame().round(4))
print(
    f"Final full-train fit: {final_fit_seconds:.3f} s; "
    f"single holdout prediction: {holdout_prediction_seconds:.4f} s."
)

## 8.1. Report-only comparison of all 30 fixed configurations

At the requested reporting stage, compare the median baseline and every
CV-selected family/preprocessing configuration. All parameters, feature
preparation and the final model were fixed using training CV before these
predictions. This extends the earlier single-model holdout protocol:
**the holdout now compares 30 configurations and is consumed for reporting**.
It is not used for another selection, parameter search, or change of winner.

Reuse the final model's stored prediction. Fit each of the other 29 entries
on the complete training split and predict the same holdout once. The
baseline predicts the median original training price. Training and CV
points come from the same fixed-configuration five-fold refits above;
error bars show fold SD. Each holdout metric is one point estimate, with
no artificial CV-style error bar. Keep the original CV order in every
chart, and mark the already selected model explicitly.

CV models train on four fifths of the training split, while holdout-report models train on all 1,168 training rows. Differences between CV and holdout therefore also reflect training size and sampling variation; the chart is not a paired significance test.


In [ ]:
if globals().get("holdout_comparison_evaluation_count", 0) != 0:
    raise RuntimeError(
        "The 30-model holdout report already exists. Reuse its predictions or restart the kernel."
    )

locked_cv_winner = best_model_name
locked_cv_order = list(leaderboard.index)
holdout_comparison_predictions = pd.DataFrame(index=y_holdout.index)
holdout_comparison_rows = []
for name in locked_cv_order:
    if name == best_model_name:
        predictions = np.asarray(y_pred_holdout)
        fit_seconds = final_fit_seconds
        prediction_seconds = holdout_prediction_seconds
    else:
        is_baseline = name == comparison_baseline_name
        report_estimator = clone(baseline if is_baseline else model_candidates[name])
        fit_started = perf_counter()
        report_estimator.fit(X, y_price if is_baseline else y)
        fit_seconds = perf_counter() - fit_started
        prediction_started = perf_counter()
        raw_predictions = report_estimator.predict(X_holdout)
        predictions = (
            np.asarray(raw_predictions)
            if is_baseline
            else inverse_predictions(raw_predictions)
        )
        prediction_seconds = perf_counter() - prediction_started
    assert len(predictions) == len(y_holdout) and np.isfinite(predictions).all()
    assert (predictions >= 0).all()
    holdout_comparison_predictions[name] = predictions
    holdout_comparison_rows.append(
        {
            "Model": name,
            "Family": model_metadata[name]["family"],
            "Part": model_metadata[name]["part"],
            "Preparation": model_metadata[name]["preparation"],
            "CV-selected final model": name == best_model_name,
            "Holdout MAE": mean_absolute_error(y_holdout, predictions),
            "Holdout RMSE": root_mean_squared_error(y_holdout, predictions),
            "Holdout R2": r2_score(y_holdout, predictions),
            "Holdout RMSLE": root_mean_squared_log_error(y_holdout, predictions),
            "Full-train fit seconds": fit_seconds,
            "Holdout prediction seconds": prediction_seconds,
        }
    )

holdout_comparison = (
    pd.DataFrame(holdout_comparison_rows).set_index("Model").loc[locked_cv_order]
)
holdout_comparison_evaluation_count = 1
assert holdout_comparison_predictions.shape == (292, 30)
np.testing.assert_array_equal(
    holdout_comparison_predictions[best_model_name], y_pred_holdout
)
assert best_model_name == locked_cv_winner
for metric in METRIC_NAMES:
    np.testing.assert_allclose(
        holdout_comparison.loc[best_model_name, f"Holdout {metric}"],
        holdout_metrics[metric],
        rtol=1e-12,
        atol=1e-12,
    )

holdout_report_table = holdout_comparison.copy()
for metric in METRIC_NAMES:
    holdout_report_table[f"Train {metric} mean"] = training_comparison_summary[
        f"Train {metric} mean"
    ]
    holdout_report_table[f"CV {metric} mean"] = training_comparison_summary[
        f"Validation {metric} mean"
    ]
display(holdout_report_table.round(5))
print(
    "Reported all 30 fixed configurations; the final CV-selected model remains:",
    best_model_name,
)

# Retain predictions so further report plots require no new holdout calls.
joblib.dump(
    {
        "predictions": holdout_comparison_predictions,
        "metrics": holdout_report_table,
        "cv_order": locked_cv_order,
        "cv_winner": locked_cv_winner,
        "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
        "experiment_source_sha256": _source_fingerprint(),
    },
    PROJECT_ROOT / ".cache" / "holdout-comparison.joblib",
)

In [ ]:
_holdout_plot_order = locked_cv_order
_holdout_plot_labels = [
    name + (" [CV winner]" if name == best_model_name else "")
    for name in _holdout_plot_order
]
_holdout_positions = np.arange(len(_holdout_plot_order))
_holdout_metric_labels = {
    "MAE": "MAE, USD (lower is better)",
    "RMSE": "RMSE, USD (lower is better)",
    "R2": "R² (higher is better)",
    "RMSLE": "RMSLE (lower is better)",
}
figure, axes = plt.subplots(2, 2, figsize=(23, 23))
for axis, metric in zip(axes.flat, METRIC_NAMES):
    cv_means = training_comparison_summary.loc[
        _holdout_plot_order, f"Validation {metric} mean"
    ]
    cv_stds = training_comparison_summary.loc[
        _holdout_plot_order, f"Validation {metric} std"
    ]
    holdout_values = holdout_comparison.loc[_holdout_plot_order, f"Holdout {metric}"]
    axis.errorbar(
        cv_means,
        _holdout_positions - 0.13,
        xerr=cv_stds,
        fmt="o",
        color="#2166ac",
        capsize=2,
        markersize=4,
        linewidth=1.1,
        label="Training CV mean ± fold SD",
    )
    axis.scatter(
        holdout_values,
        _holdout_positions + 0.13,
        marker="D",
        color="#d95f02",
        s=25,
        label="Single holdout point",
        zorder=3,
    )
    for position, cv_value, holdout_value in zip(
        _holdout_positions, cv_means, holdout_values
    ):
        axis.plot(
            [cv_value, holdout_value],
            [position - 0.13, position + 0.13],
            color="#bbbbbb",
            linewidth=0.6,
            zorder=0,
        )
    selected_position = _holdout_plot_order.index(best_model_name)
    axis.axhspan(
        selected_position - 0.45,
        selected_position + 0.45,
        color="#ddeedb",
        alpha=0.7,
        zorder=-1,
    )
    axis.set_yticks(_holdout_positions, _holdout_plot_labels, fontsize=8.5)
    axis.invert_yaxis()
    axis.set_xlabel(_holdout_metric_labels[metric], fontsize=11)
    axis.set_title(f"CV versus holdout: {metric}", fontsize=14)
    axis.grid(axis="x", alpha=0.25)
    if metric in ("MAE", "RMSE"):
        axis.ticklabel_format(axis="x", style="plain", useOffset=False)
    axis.legend(fontsize=9, loc="best")
figure.suptitle(
    "All 30 configurations — holdout reporting in the original CV order",
    fontsize=19,
    y=0.997,
)
figure.tight_layout(rect=(0, 0, 1, 0.975), h_pad=2.0, w_pad=3.0)
plt.show()

figure, axis = plt.subplots(figsize=(15, 17))
for offset, prefix, color, marker, label in [
    (-0.20, "Train", "#359f58", "o", "Training mean ± fold SD"),
    (0.0, "Validation", "#2166ac", "o", "Validation mean ± fold SD"),
]:
    axis.errorbar(
        training_comparison_summary.loc[_holdout_plot_order, f"{prefix} RMSLE mean"],
        _holdout_positions + offset,
        xerr=training_comparison_summary.loc[
            _holdout_plot_order, f"{prefix} RMSLE std"
        ],
        fmt=marker,
        color=color,
        capsize=2,
        markersize=4,
        linewidth=1.1,
        label=label,
    )
axis.scatter(
    holdout_comparison.loc[_holdout_plot_order, "Holdout RMSLE"],
    _holdout_positions + 0.20,
    color="#d95f02",
    marker="D",
    s=28,
    label="Holdout point estimate",
    zorder=3,
)
selected_position = _holdout_plot_order.index(best_model_name)
axis.axhspan(
    selected_position - 0.45,
    selected_position + 0.45,
    color="#ddeedb",
    alpha=0.7,
    zorder=-1,
)
axis.set_yticks(_holdout_positions, _holdout_plot_labels, fontsize=10)
axis.invert_yaxis()
axis.set_xlabel("RMSLE — lower is better")
axis.set_title(
    "Train, training CV, and holdout for all 30 fixed configurations", fontsize=15
)
axis.grid(axis="x", alpha=0.25)
axis.legend(loc="upper right", fontsize=10)
figure.tight_layout()
plt.show()

# 9. Residual analysis of the final model

Residual = actual price − predicted price: positive means underprediction. Dollar errors emphasize expensive houses, while relative and log errors reveal large proportional mistakes. Every plot and table below reuses the one stored holdout prediction array. Observations may motivate a future experiment with a new evaluation protocol; they do not trigger another model fit or a second holdout assessment here.


In [ ]:
residuals = y_holdout - y_pred_holdout
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.scatterplot(x=y_holdout, y=y_pred_holdout, alpha=0.6, ax=axes[0])
limits = [0, max(y_holdout.max(), y_pred_holdout.max()) * 1.05]
axes[0].plot(limits, limits, "--", color="red")
axes[0].set(
    xlim=limits,
    ylim=limits,
    xlabel="Actual price, USD",
    ylabel="Predicted price, USD",
    title="Actual vs. predicted price",
)
sns.histplot(residuals, bins=30, kde=True, ax=axes[1])
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set(
    xlabel="Actual − predicted price, USD",
    ylabel="Number of houses",
    title="Residual distribution",
)
sns.scatterplot(x=y_pred_holdout, y=residuals, alpha=0.6, ax=axes[2])
axes[2].axhline(0, color="red", linestyle="--")
axes[2].set(
    xlabel="Predicted price, USD",
    ylabel="Residual, USD",
    title="Residuals vs. predictions",
)
for ax in axes:
    ax.grid(alpha=0.2)
fig.suptitle(f"One holdout evaluation: {best_model_name}", y=1.02)
plt.tight_layout()
plt.show()

error_analysis = test.copy()
error_analysis.insert(0, "Id", ids.loc[error_analysis.index])
error_analysis["Prediction"] = pd.Series(y_pred_holdout, index=y_holdout.index)
error_analysis["Residual"] = error_analysis["SalePrice"] - error_analysis["Prediction"]
error_analysis["AbsError"] = error_analysis["Residual"].abs()
error_analysis["AbsPctError"] = (
    100 * error_analysis["AbsError"] / error_analysis["SalePrice"]
)
error_analysis["LogResidual"] = np.log1p(error_analysis["SalePrice"]) - np.log1p(
    error_analysis["Prediction"]
)
error_analysis["AbsLogError"] = error_analysis["LogResidual"].abs()
error_columns = [
    "Id",
    "SalePrice",
    "Prediction",
    "Residual",
    "AbsError",
    "AbsPctError",
    "AbsLogError",
    "GrLivArea",
    "TotalBsmtSF",
    "OverallQual",
    "YearBuilt",
    "Neighborhood",
    "SaleType",
    "SaleCondition",
]
worst_houses = error_analysis.nlargest(10, "AbsError")
for error_measure, title in [
    ("AbsError", "Ten largest absolute dollar errors"),
    ("AbsPctError", "Ten largest absolute percentage errors"),
    ("AbsLogError", "Ten largest absolute log errors"),
]:
    display(Markdown(f"**{title}**"))
    display(error_analysis.nlargest(10, error_measure)[error_columns].round(3))

## 9.1. Error patterns and concrete houses

Price-band summaries describe errors at different actual prices; their boundaries come only from this final descriptive analysis. Neighborhood and sale-condition tables always show group counts, and groups smaller than ten are flagged. Small groups and the ten worst errors cannot establish systematic causes.

Feature meanings follow Kaggle’s `data_description.txt`: `GrLivArea` is above-grade living area in square feet, `TotalBsmtSF` is basement area, `OverallQual` rates materials and finish from 1 to 10, `YearBuilt` is the original construction date, and `Neighborhood` identifies a location within Ames. Sale-condition labels describe the transaction, not a direct inspection of a house's physical condition.


In [ ]:
def summarize_residual_groups(data, group_column):
    summary = data.groupby(group_column, observed=True, dropna=False).agg(
        Houses=("Residual", "size"),
        MeanResidualUSD=("Residual", "mean"),
        MAE_USD=("AbsError", "mean"),
        RMSE_USD=("Residual", lambda residual: np.sqrt(np.mean(np.square(residual)))),
        MedianAbsolutePercentError=("AbsPctError", "median"),
    )
    summary["Fewer than 10 houses"] = summary["Houses"] < 10
    return summary


error_analysis["Actual price band"] = pd.qcut(
    error_analysis["SalePrice"], q=4, duplicates="drop"
)
display(summarize_residual_groups(error_analysis, "Actual price band").round(2))
display(
    summarize_residual_groups(error_analysis, "Neighborhood")
    .sort_values("MAE_USD", ascending=False)
    .round(2)
)
display(
    summarize_residual_groups(error_analysis, "SaleCondition")
    .sort_values("MAE_USD", ascending=False)
    .round(2)
)

sale_condition_meanings = {
    "Normal": "normal sale",
    "Abnorml": "abnormal transaction such as a trade, foreclosure, or short sale",
    "AdjLand": "adjoining land purchase",
    "Alloca": "allocation involving linked properties with separate deeds",
    "Family": "sale between family members",
    "Partial": "home incomplete when last assessed",
}
summary_lines = [
    f"The mean holdout residual for **{best_model_name}** is "
    f"**${residuals.mean():+,.2f}**; the median is **${residuals.median():+,.2f}**. "
    f"The model underpredicts **{int((residuals > 0).sum())}/{len(residuals)}** houses.",
    "",
    "The three largest dollar errors, with comparisons restricted to training data:",
    "",
]
for house in worst_houses.head(3).itertuples():
    direction = "underprediction" if house.Residual > 0 else "overprediction"
    area_percentile = 100 * df["GrLivArea"].le(house.GrLivArea).mean()
    price_percentile = 100 * y_price.le(house.SalePrice).mean()
    neighborhood_count = int(df["Neighborhood"].eq(house.Neighborhood).sum())
    condition_count = int(df["SaleCondition"].eq(house.SaleCondition).sum())
    quality_count = int(df["OverallQual"].eq(house.OverallQual).sum())
    comparable_mask = (
        df["Neighborhood"].eq(house.Neighborhood)
        & df["OverallQual"].between(house.OverallQual - 1, house.OverallQual + 1)
        & df["GrLivArea"].between(house.GrLivArea * 0.75, house.GrLivArea * 1.25)
    )
    comparable_prices = y_price.loc[comparable_mask]
    if len(comparable_prices):
        comparable_text = (
            f"{len(comparable_prices)} training houses match the same neighborhood, "
            f"quality within one point, and above-grade area within 25%; their price "
            f"median is ${comparable_prices.median():,.0f} "
            f"(IQR ${comparable_prices.quantile(0.25):,.0f}–${comparable_prices.quantile(0.75):,.0f})."
        )
    else:
        comparable_text = (
            "No training house matches the same neighborhood, quality within one point, "
            "and above-grade area within 25%."
        )
    hypotheses = []
    if price_percentile >= 95 and house.Residual > 0:
        hypotheses.append(
            "Its price is in the expensive training tail; limited expensive comparables "
            "and shrinkage toward more common prices are plausible contributors."
        )
    if area_percentile >= 95:
        hypotheses.append(
            "Its above-grade area is unusually large, and size alone may not capture "
            "the value of its quality or transaction circumstances."
        )
    if neighborhood_count < 10 or condition_count < 10 or len(comparable_prices) < 5:
        hypotheses.append(
            "Sparse comparable training observations may make this case harder to predict."
        )
    if house.SaleCondition != "Normal":
        hypotheses.append(
            "Its nonstandard sale condition may indicate pricing circumstances beyond typical sales."
        )
    if not hypotheses:
        hypotheses.append(
            "The observed attributes alone do not identify the source of the error; "
            "unrecorded condition or transaction details remain possible explanations."
        )
    summary_lines.extend(
        [
            f"- **Id {int(house.Id)}:** actual **${house.SalePrice:,.0f}**, "
            f"predicted **${house.Prediction:,.0f}**, {direction} of "
            f"**${house.AbsError:,.0f} ({house.AbsPctError:.1f}%)**. "
            f"Above-grade area **{house.GrLivArea:,.0f} ft²** "
            f"(training percentile {area_percentile:.1f}), basement **{house.TotalBsmtSF:,.0f} ft²**, "
            f"materials/finish quality **{house.OverallQual:g}/10** "
            f"({quality_count} train houses at that score), built **{int(house.YearBuilt)}**. "
            f"Neighborhood **{house.Neighborhood}** has **{neighborhood_count}** train houses; "
            f"**{house.SaleCondition}** means "
            f'"{sale_condition_meanings.get(house.SaleCondition, str(house.SaleCondition))}" '
            f"and occurs **{condition_count}** times in train. "
            f"{comparable_text} {' '.join(hypotheses)}",
            "",
        ]
    )
summary_lines.append(
    "These are case-specific hypotheses, not established causes. The largest dollar "
    "errors need not be the largest proportional errors. The holdout has now been "
    "consumed for final reporting; no choice of model or parameters is revised from it."
)
display(Markdown("\n".join(summary_lines)))

# 10. Reproducibility checks

The checks validate the common result schema, prediction alignment, finite diagnostics, and the single 30-configuration holdout reporting pass. Model selection remains the locked training-CV ranking. A small machine-readable summary is cached for synchronizing the README with this execution; the notebook's displayed results remain authoritative.


In [ ]:
from pathlib import Path
from scipy import sparse

assert len(leaderboard) == 30, (
    "Expected baseline, 12 Part I entries, and 17 Part II entries."
)
assert len(model_candidates) == 29
assert len(cv_splits) == 5
assert holdout_evaluation_count == 1
assert holdout_comparison_evaluation_count == 1
assert holdout_comparison_predictions.shape == (292, 30)
assert set(holdout_comparison_predictions.columns) == set(leaderboard.index)
assert training_comparison_summary.shape[0] == 30
assert np.isfinite(holdout_comparison_predictions.to_numpy()).all()
assert best_model_name == locked_cv_winner
assert best_model_name == min(
    model_candidates, key=lambda name: (cv_folds[name]["RMSLE"].mean(), name)
)
assert len(X) + len(X_holdout) == 1460
assert X.index.is_unique and X_holdout.index.is_unique
assert X.index.intersection(X_holdout.index).empty
assert np.isfinite(y_pred_holdout).all() and (y_pred_holdout >= 0).all()
assert np.isfinite(final_oof_predictions).all()
assert np.isfinite(holdout_metrics.to_numpy()).all()
assert set(permutation_table.index) == set(X.columns)
assert permutation_array.shape == (5, X.shape[1], 10)
assert np.isfinite(permutation_table.to_numpy()).all()
for name, folds in cv_folds.items():
    assert list(folds.columns) == METRIC_NAMES and len(folds) == 5
    assert np.isfinite(folds.to_numpy()).all()
    assert len(cv_fit_times[name]) == 5 and np.isfinite(cv_fit_times[name]).all()
    assert (np.asarray(cv_fit_times[name]) >= 0).all()
if native_importance is not None:
    assert np.isfinite(native_importance).all()
    assert np.isclose(native_importance.sum(), 1.0) or np.isclose(
        native_importance.sum(), 0.0
    )

transformed_preview = final_model.named_steps["preparation"].transform(X.iloc[:8])
if sparse.issparse(transformed_preview):
    assert np.isfinite(transformed_preview.data).all()
elif isinstance(transformed_preview, pd.DataFrame):
    assert np.isfinite(
        transformed_preview.select_dtypes(include="number").to_numpy()
    ).all()
else:
    assert np.isfinite(np.asarray(transformed_preview, dtype=float)).all()

final_results = {
    "selected_model": best_model_name,
    "selected_metadata": model_metadata[best_model_name],
    "best_part_i": best_part_i_name,
    "best_part_ii": best_part_ii_name,
    "relative_cv_rmsle_improvement_percent": float(relative_cv_gain),
    "cv_fit_time_ratio_part_ii_over_i": float(fit_time_ratio),
    "cv_metrics": {
        metric: {
            "mean": float(cv_folds[best_model_name][metric].mean()),
            "std": float(cv_folds[best_model_name][metric].std(ddof=1)),
        }
        for metric in METRIC_NAMES
    },
    "holdout_metrics": {
        metric: float(value) for metric, value in holdout_metrics.items()
    },
    "final_fit_seconds": float(final_fit_seconds),
    "holdout_prediction_seconds": float(holdout_prediction_seconds),
    "holdout_evaluation_count": int(holdout_evaluation_count),
    "holdout_comparison_models": int(holdout_comparison.shape[0]),
    "holdout_comparison_passes": int(holdout_comparison_evaluation_count),
    "holdout_reporting_protocol": "All 30 fixed CV configurations; final model unchanged",
    "validation": {
        "status": "passed",
        "cv_entries": int(len(leaderboard)),
        "trained_candidates": int(len(model_candidates)),
        "holdout_evaluations": int(holdout_evaluation_count),
        "holdout_models_reported": int(holdout_comparison.shape[0]),
        "training_comparison_models": int(training_comparison_summary.shape[0]),
        "permutation_array_shape": list(permutation_array.shape),
        "native_importance_available": native_importance is not None,
        "final_cv_refit_metrics_reproduced": True,
    },
}
results_path = PROJECT_ROOT / ".cache" / "final_results.json"
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(json.dumps(final_results, indent=2, default=str) + "\n")
print(
    "Validated: 30 CV entries, aligned OOF predictions, finite importance, and one report-only holdout pass for all 30 fixed configurations."
)

# 11. Final conclusions — training pipeline, model and optimization

The comparisons below answer three separate questions: which feature-preparation
pipeline is best on training CV, which model/configuration is carried forward,
and which search produced that configuration. Selection uses **training CV**.
A low resubstitution training error alone is not the criterion for the best
pipeline. The report-only holdout comparison does not change the winner.

In [ ]:
pipeline_descriptions = {
    "base": "Original features",
    "FE": "Original + nine engineered features",
    "high_corr": "Engineered features + supervised numeric Spearman selection",
}
pipeline_conclusion_rows = []
for preparation in ["base", "FE", "high_corr"]:
    subset = leaderboard[leaderboard["Preparation"].eq(preparation)]
    assert not subset.empty
    winner_name = min(
        subset.index, key=lambda name: (subset.loc[name, "RMSLE mean"], name)
    )
    row = leaderboard.loc[winner_name]
    pipeline_conclusion_rows.append(
        {
            "Train preparation": preparation,
            "Feature pipeline": pipeline_descriptions[preparation],
            "Best model on this preparation": winner_name,
            "Categories": model_metadata[winner_name]["encoding"],
            "CV RMSLE mean": float(row["RMSLE mean"]),
            "CV RMSLE sample std": float(row["RMSLE std"]),
            "CV MAE, USD": float(row["MAE mean"]),
        }
    )
pipeline_conclusions = pd.DataFrame(pipeline_conclusion_rows).set_index(
    "Train preparation"
)
assert (
    np.isfinite(
        pipeline_conclusions[["CV RMSLE mean", "CV RMSLE sample std", "CV MAE, USD"]]
    )
    .all()
    .all()
)
display(pipeline_conclusions.sort_values("CV RMSLE mean").round(5))

selected_searches = []
for method in SEARCH_METHODS:
    payload = ensemble_search_results[(best_model_name, method)]
    records = sorted(payload["records"], key=lambda record: record["sequence"])
    best_record = next(
        record for record in records if record["config_id"] == payload["best_config_id"]
    )
    score = best_record["mean_rmsle"]
    first_position = next(
        i + 1
        for i, record in enumerate(records)
        if np.isclose(record["mean_rmsle"], score, rtol=0, atol=1e-12)
    )
    selected_searches.append(
        {
            "Optimization method": method,
            "Best CV RMSLE": score,
            "CV RMSLE sample std": best_record["folds"]["RMSLE"].std(ddof=1),
            "Total search wall seconds": payload["search_seconds"],
            "First optimum position in recorded unique order": first_position,
        }
    )
selected_optimization_conclusions = pd.DataFrame(selected_searches).set_index(
    "Optimization method"
)
assert np.ptp(selected_optimization_conclusions["Best CV RMSLE"]) <= 1e-12
display(selected_optimization_conclusions.round(5))

selected_reference = ensemble_search_results[(best_model_name, "GridSearchCV")]
selected_reference_record = next(
    record
    for record in selected_reference["records"]
    if record["config_id"] == selected_reference["best_config_id"]
)
selected_parameters = {
    name.removeprefix("regressor__"): value
    for name, value in selected_reference_record["params"].items()
}
assert selected_parameters == model_metadata[best_model_name]["parameters"]
selected_metadata = model_metadata[best_model_name]
grid_seconds = selected_optimization_conclusions.loc[
    "GridSearchCV", "Total search wall seconds"
]
random_seconds = selected_optimization_conclusions.loc[
    "RandomizedSearchCV", "Total search wall seconds"
]
tpe_seconds = selected_optimization_conclusions.loc[
    "Optuna TPE + completion", "Total search wall seconds"
]
gp_seconds = selected_optimization_conclusions.loc[
    "Bayesian GP + EI", "Total search wall seconds"
]
grid_random_difference = abs(grid_seconds - random_seconds)
grid_random_difference_percent = (
    100 * grid_random_difference / min(grid_seconds, random_seconds)
)
conclusion_lines = [
    f"**Best training pipeline: {selected_metadata['preparation']} + "
    f"{selected_metadata['encoding']} categories. Best model: {best_model_name}.**",
    "",
    "The FE preparation (`model`) adds the nine house features, learns the "
    "86% dominant-value filter inside each training fold, median-imputes and scales "
    "numeric features with missing indicators, and uses a `Missing` category plus "
    "one-hot encoding for categorical features. The regressor predicts `log1p(SalePrice)`.",
    "",
    f"This pipeline has **mean five-fold CV RMSLE "
    f"{leaderboard.loc[best_model_name, 'RMSLE mean']:.5f} ± "
    f"{leaderboard.loc[best_model_name, 'RMSLE std']:.5f}**, with "
    f"**CV MAE ${leaderboard.loc[best_model_name, 'MAE mean']:,.2f}**. "
    "The table above compares the best model from each preparation branch.",
    "",
    f"**Selected parameters:** `{json.dumps(selected_parameters, sort_keys=True)}`. "
    "CatBoost additionally uses Bernoulli bootstrap, CPU execution and seed 42.",
    "",
    "**Optimization method used to carry the selected configuration forward: "
    "GridSearchCV.** The implementation explicitly takes its reference winner. "
    "RandomizedSearchCV, Optuna TPE with exhaustive completion, and GP + EI "
    "independently recovered the same parameters and exactly the same fold scores. "
    "**There is no unique optimization-method winner in final predictive quality.**",
    "",
    f"The original measured total search costs for this model are "
    f"**{grid_seconds:.2f} s (Grid)**, **{random_seconds:.2f} s (Randomized)**, "
    f"**{tpe_seconds:.2f} s (TPE + completion)** and **{gp_seconds:.2f} s (GP + EI)**. "
    f"The Grid/Randomized difference is **{grid_random_difference:.3f} s "
    f"({grid_random_difference_percent:.2f}%)** in one measurement each; "
    "this is not evidence of an intrinsic speed advantage. "
    "GridSearchCV is a straightforward choice for this deliberately small 32-point "
    "space; the adaptive methods have no remaining quality advantage once every "
    "configuration must be tested.",
    "",
    "The recorded optimum positions and wall-time differences require care: "
    "SearchCV records parameter-list order rather than parallel completion timestamps. "
    "Grid/Randomized can schedule across eight workers, whereas the sequential "
    "adaptive searches have at most five fold fits active for one candidate. "
    "Thus measured wall time reflects the implementation, CPU scheduling and "
    "optimizer overhead together. These results do not establish which optimizer "
    "would win under a smaller budget or a larger search space.",
]
display(Markdown(chr(10).join(conclusion_lines)))

# References

- [Official competition data and feature description](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data).
- [scikit-learn: RandomizedSearchCV and sampling without replacement](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html).
- [Optuna: reproducible optimization](https://optuna.readthedocs.io/en/stable/faq.html).
- [scikit-optimize: Expected Improvement](https://scikit-optimize.readthedocs.io/en/stable/modules/generated/skopt.acquisition.gaussian_ei.html).
- [CatBoost: categorical feature processing](https://catboost.ai/docs/en/concepts/algorithm-main-stages_cat-to-numberic) and [Bernoulli bootstrap](https://catboost.ai/docs/en/concepts/algorithm-main-stages_bootstrap-options).
- [LightGBM: subsampling and importance settings](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRegressor.html).
- [XGBoost: regressor API and total-gain importance](https://xgboost.readthedocs.io/en/stable/python/python_api.html).
- [scikit-learn: permutation importance, correlation and impurity-importance limitations](https://scikit-learn.org/stable/modules/permutation_importance.html).
